# Confidence-gated TTA (3) and saliency-masked second pass (4) on the 0.915 model (no training)

Runtime > any GPU (A100 fastest), then Run all. A few minutes.

- **3. Gated TTA:** images whose current top probability is below a threshold get the average over 26 views (13 shifts of 0-2 px x mirror); confident images keep the current prediction.
- **4. Saliency mask:** an input-gradient saliency map from the first pass; a second pass on the image with its least salient pixels set to the training mean; final logits = (1 - w) * original + w * masked.
- Settings (threshold; keep fraction and w) are chosen on **val** and judged by 2-fold cross-fitting on val; internal test is printed as the honest check. A submission is written only if one beats the current setting on val by > 0.2 pt (5 of 2,500). The hidden test images are only predicted; nothing is learned from them.


## 1. Paths, Drive, device

In [2]:
# ================= Confidence-gated TTA and saliency-masked second pass on the 0.915 model (no training) =================
import os, re, glob, time, json
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

# ---- paths: set RUN_NAME if you know it; otherwise the run whose submission has internal test 0.9812 is found automatically
RUN_NAME = None                       # e.g. "kd_sewrn_only_student_..." (the 0.915 run)
FIND_TEST_TAG = "test0.9812"          # used only when RUN_NAME is None
try:
    import google.colab  # noqa
    IN_COLAB = True
    from google.colab import drive; drive.mount("/content/drive")
    ROOT, WORK_DIR = "/content/drive/MyDrive/cifar10_resnet_challenge", "/content/work"
except ImportError:
    IN_COLAB = False
    ROOT = os.environ.get("TTA_ROOT", "/scratch/ds8223/cifar10_resnet_challenge"); WORK_DIR = os.environ.get("TTA_WORK", "/scratch/ds8223/kd_work")
CACHE_DIR, SPLIT_DIR = os.path.join(ROOT, "data", "cache"), os.path.join(ROOT, "data", "splits")

if RUN_NAME is None:
    hits = sorted({os.path.basename(os.path.dirname(p)) for p in glob.glob(os.path.join(ROOT, "runs", "*", f"submission_*{FIND_TEST_TAG}*.csv"))})
    print("runs with a submission tagged", FIND_TEST_TAG, ":", hits)
    assert len(hits) == 1, "found 0 or several runs: set RUN_NAME by hand (see the list above)"
    RUN_NAME = hits[0]
RUN_DIR = os.path.join(ROOT, "runs", RUN_NAME)
BEST_CKPT = os.path.join(RUN_DIR, "best.pt")
print("run:", RUN_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (slow: use a GPU)")
AMP_DTYPE = torch.bfloat16 if device.type == "cpu" or torch.cuda.is_bf16_supported() else None

Mounted at /content/drive
runs with a submission tagged test0.9812 : ['kd_sewrn_only_student_resnext29_slasd_400ep_v1']
run: kd_sewrn_only_student_resnext29_slasd_400ep_v1
device: NVIDIA A100-SXM4-40GB


## 2. Model code

In [3]:
# ---- Models (same code as the training notebooks): ResNet, ResNeXt-29, Shake-Shake, chosen with CFG["ARCH"] ----
class BasicBlock(nn.Module):
    """Post-activation basic block (our original ResNet)."""
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Identity()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x), inplace=True)

class CifarResNet(nn.Module):
    def __init__(self, widths=(64, 128, 256), blocks=(3, 3, 3), num_classes=10, dropout=0.0):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        layers, cin = [], widths[0]
        for i, (w, nb) in enumerate(zip(widths, blocks)):
            for j in range(nb):
                layers.append(BasicBlock(cin, w, 2 if (j == 0 and i > 0) else 1)); cin = w
        self.layers = nn.Sequential(*layers)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        for b in self.layers: nn.init.zeros_(b.bn2.weight)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)

class ResNeXtBlock(nn.Module):
    """ResNeXt bottleneck: 1x1 conv -> grouped 3x3 conv (cardinality groups) -> 1x1 conv, plus skip (Xie et al. 2017)."""
    def __init__(self, cin, inner, cout, stride, cardinality):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, inner, 1, bias=False); self.bn1 = nn.BatchNorm2d(inner)
        self.conv2 = nn.Conv2d(inner, inner, 3, stride, 1, groups=cardinality, bias=False); self.bn2 = nn.BatchNorm2d(inner)
        self.conv3 = nn.Conv2d(inner, cout, 1, bias=False); self.bn3 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Identity()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = F.relu(self.bn2(self.conv2(out)), inplace=True)
        out = self.bn3(self.conv3(out))
        return F.relu(out + self.shortcut(x), inplace=True)

class ResNeXt29(nn.Module):
    """CIFAR ResNeXt-29: 3x3 stem (64) + 3 stages x 3 blocks; output widths 256/512/1024;
    inner width = cardinality * bottleneck_width, doubled at each stage. (2, 64) is the standard 2x64d."""
    def __init__(self, cardinality=2, bottleneck_width=64, blocks=(3, 3, 3), out_widths=(256, 512, 1024),
                 num_classes=10, dropout=0.0, stem=64, strides=(1, 2, 2)):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, stem, 3, 1, 1, bias=False), nn.BatchNorm2d(stem), nn.ReLU(inplace=True))
        layers, cin = [], stem
        for i, (cout, nb) in enumerate(zip(out_widths, blocks)):
            inner = cardinality * bottleneck_width * (2 ** i)
            for j in range(nb):
                layers.append(ResNeXtBlock(cin, inner, cout, strides[i] if j == 0 else 1, cardinality)); cin = cout
        self.layers = nn.Sequential(*layers)
        self.drop = nn.Dropout(dropout); self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        for b in self.layers: nn.init.zeros_(b.bn3.weight)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)

class SE(nn.Module):
    """Squeeze-excitation (Hu et al. 2018): channel weights from global average pooling. Teachers only."""
    def __init__(self, c, r=16):
        super().__init__()
        self.fc1, self.fc2 = nn.Conv2d(c, c // r, 1), nn.Conv2d(c // r, c, 1)
    def forward(self, x):
        return x * torch.sigmoid(self.fc2(F.relu(self.fc1(F.adaptive_avg_pool2d(x, 1)))))

class WBlock(nn.Module):
    """Pre-activation wide residual block (Zagoruyko & Komodakis 2016)."""
    def __init__(self, cin, cout, stride, drop, se_r=0):
        super().__init__()
        self.bn1, self.conv1 = nn.BatchNorm2d(cin), nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn2, self.conv2 = nn.BatchNorm2d(cout), nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.dropout = nn.Dropout(drop)
        self.skip = None if (stride == 1 and cin == cout) else nn.Conv2d(cin, cout, 1, stride, bias=False)
        if se_r: self.se = SE(cout, se_r)
    def forward(self, x):
        o = F.relu(self.bn1(x))
        sc = x if self.skip is None else self.skip(o)
        o = self.conv2(self.dropout(F.relu(self.bn2(self.conv1(o)))))
        if hasattr(self, "se"): o = self.se(o)
        return o + sc

class WRN(nn.Module):
    """WideResNet for CIFAR (Zagoruyko & Komodakis 2016). k = widen factor (WRN-28-10: 36.5M params) or explicit stage widths."""
    def __init__(self, depth=28, k=10, dropout=0.0, num_classes=10, se_r=0):
        super().__init__()
        n = (depth - 4) // 6
        w = [16] + (list(k) if isinstance(k, (tuple, list)) else [16 * k, 32 * k, 64 * k])   # k: widen factor, or the 3 stage widths
        self.stem = nn.Conv2d(3, w[0], 3, 1, 1, bias=False)
        layers, cin = [], w[0]
        for i in range(3):
            for j in range(n):
                layers.append(WBlock(cin, w[i + 1], 2 if (j == 0 and i > 0) else 1, dropout, se_r)); cin = w[i + 1]
        self.layers = nn.Sequential(*layers); self.bn = nn.BatchNorm2d(cin)
        self.drop = nn.Dropout(0.0); self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(F.relu(self.bn(self.layers(self.stem(x)))), 1), 1)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))

def _build_base(c):
    """Build any of our architectures from a config dict (also rebuilds teachers from their saved cfg)."""
    arch = c.get("ARCH", "resnet")
    if arch == "resnet":
        return CifarResNet(c["WIDTHS"], c["BLOCKS"], dropout=c.get("DROPOUT", 0.0))
    if arch == "resnext":
        return ResNeXt29(c.get("CARDINALITY", 2), c.get("BOTTLENECK_WIDTH", 64), c.get("BLOCKS", (3, 3, 3)),
                         c.get("WIDTHS", (256, 512, 1024)), dropout=c.get("DROPOUT", 0.0),
                         strides=tuple(c.get("STRIDES", (1, 2, 2))))   # (1, 1, 2) = higher-resolution student, same params
    if arch in ("wrn", "sewrn"):     # sewrn = WRN with SE blocks: a training-only teacher
        return WRN(c["DEPTH"], c["WIDEN"], c.get("DROPOUT", 0.0), se_r=c.get("SE_REDUCTION", 16) if arch == "sewrn" else 0)
    raise ValueError(f"unknown ARCH {arch}")

def build_model(c):
    """Base network; with SLA on, adds the joint (class x rotation) head used only during training."""
    net = _build_base(c)
    if c.get("SLA", False):
        net.fc_joint = nn.Linear(net.fc.in_features, 10 * c.get("SLA_ROTATIONS", 4))
    return net

def forward_both(net, x):
    """Returns (10-class self-distillation head logits, joint class x rotation head logits)."""
    f = net.drop(net.features(x))
    return net.fc(f), net.fc_joint(f)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 3. Load the 0.915 best.pt, data and split

In [4]:
# ---- checkpoint, data, split (same split file as every run)
best = torch.load(BEST_CKPT, map_location=device, weights_only=False)
CFG = best["cfg"]
X_full = np.load(os.path.join(CACHE_DIR, "train_x.npy")); y_full = np.load(os.path.join(CACHE_DIR, "train_y.npy"))
X_test = np.load(os.path.join(CACHE_DIR, "test_x.npy")); test_ids = np.load(os.path.join(CACHE_DIR, "test_ids.npy"))
SAMPLE_SUB = pd.read_csv(os.path.join(CACHE_DIR, "sample_submission.csv"))
sp = np.load(os.path.join(SPLIT_DIR, f"split_seed{CFG['SPLIT_SEED']}_val{CFG['VAL_FRAC']}_test{CFG['TEST_FRAC']}_n{len(y_full)}.npz"))
va_idx, te_idx = sp["val"], sp["test"]
def to_dev(x): return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous().to(device)
Xva, yva = to_dev(X_full[va_idx]), torch.from_numpy(y_full[va_idx]).to(device)
Xte_int, yte_int = to_dev(X_full[te_idx]), torch.from_numpy(y_full[te_idx]).to(device)
Xkaggle = to_dev(X_test)
MEAN, STD = best["norm"]["mean"].to(device), best["norm"]["std"].to(device)   # exact normalisation of that run
def normalize(x_u8): return ((x_u8.float() / 255 - MEAN) / STD).contiguous(memory_format=torch.channels_last)

def _ag_logits(m, xb):
    """SLA+AG: average the joint-head logit of (class, k) on the image rotated by k*90 degrees."""
    agg = 0
    for k in range(CFG.get("SLA_ROTATIONS", 4)):
        _, j = forward_both(m, torch.rot90(xb, k, dims=(2, 3)))
        agg = agg + j.float().view(len(xb), 10, -1)[:, :, k]
    return agg / CFG.get("SLA_ROTATIONS", 4)

net = build_model(CFG).to(device).to(memory_format=torch.channels_last)
net.load_state_dict(best["model"]); net.eval()
has_joint = getattr(net, "fc_joint", None) is not None
print(f"params {count_params(net):,} | saved val {best['val_acc']:.4f} | SLA head: {has_joint} | val {len(Xva)} | internal test {len(Xte_int)} | kaggle {len(Xkaggle)}")

params 4,998,738 | saved val 0.9856 | SLA head: True | val 2500 | internal test 2500 | kaggle 2000


## 4. Gated TTA and saliency mask, decided on val

In [5]:
# ---- shared pieces ----
def shift(x, dy, dx, p=2):
    if dy == 0 and dx == 0: return x
    xp = F.pad(x, (p, p, p, p), mode="reflect")
    return xp[:, :, p - dy:p - dy + 32, p - dx:p - dx + 32].contiguous(memory_format=torch.channels_last)
def ac(): return torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None)
acc = lambda s, y: (s.argmax(1) == y).float().mean().item()
g = torch.Generator().manual_seed(0); idx = torch.randperm(len(yva), generator=g).to(device); h = len(idx) // 2
FOLDS = [(idx[:h], idx[h:]), (idx[h:], idx[:h])]

def crossfit(score_fn, grid):
    """Choose the setting on one half of val (by accuracy), score it on the other half; returns honest val accuracy."""
    correct = 0
    for a, b in FOLDS:
        best = max(grid, key=lambda p: acc(score_fn("val", p)[a], yva[a]))
        correct += (score_fn("val", best)[b].argmax(1) == yva[b]).sum().item()
    return correct / len(yva), max(grid, key=lambda p: acc(score_fn("val", p), yva))

DATA = {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}
t0 = time.time()

# ---- current setting (exactly the 0.915 submission): sum of softmax over image + mirror ----
@torch.no_grad()
def base_probs(X, bs=500):
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs])
        with ac(): out.append(net(xb).float().softmax(1) + net(xb.flip(3)).float().softmax(1))
    return torch.cat(out) / 2
P0 = {k: base_probs(X) for k, X in DATA.items()}
print(f"CURRENT: val {acc(P0['val'], yva):.4f} | internal test {acc(P0['test'], yte_int):.4f}   (expected 0.9852 / 0.9812)")

# ======================= 3. Confidence-gated TTA =======================
# Images whose current top probability is below a threshold get the mean over 18 views (image + mirror, shifts of 1-2 px);
# confident images keep their current prediction. Threshold chosen on val (cross-fitted).
VIEWS = [(dy, dx) for dy in (-2, -1, 0, 1, 2) for dx in (-2, -1, 0, 1, 2) if abs(dy) + abs(dx) <= 2]   # 13 positions
@torch.no_grad()
def tta_probs(X, bs=500):
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs]); p = 0
        for dy, dx in VIEWS:
            for v in (xb, xb.flip(3)):
                with ac(): p = p + net(shift(v, dy, dx)).float().softmax(1)
        out.append(p / (2 * len(VIEWS)))
    return torch.cat(out)
PT = {k: tta_probs(X) for k, X in DATA.items()}
def gated(split, thr):
    conf = P0[split].max(1).values
    return torch.where((conf < thr)[:, None], PT[split], P0[split])
GRID3 = [0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 0.95, 1.01]          # 1.01 = TTA on every image
cv3, thr3 = crossfit(gated, GRID3)
n_unc = int((P0["val"].max(1).values < thr3).sum())
print(f"\n3. GATED TTA: cross-fitted val {cv3:.4f} | threshold chosen on all val {thr3} ({n_unc} of {len(yva)} val images below it) "
      f"| internal test {acc(gated('test', thr3), yte_int):.4f}")

# ======================= 4. Saliency-masked second pass =======================
# Pass 1: input gradient of the predicted class -> saliency per pixel (|grad| summed over channels, 3x3 smoothed).
# Pass 2: the image with its LEAST salient pixels set to the training mean (0 after normalisation).
# Final logits = (1 - w) * original + w * masked; keep fraction and w chosen on val (cross-fitted). Done for image and mirror.
def masked_logits(X, keep, bs=250):
    out_o, out_m = [], []
    for i in range(0, len(X), bs):
        lo, lm = 0, 0
        for flip in (False, True):
            xb = normalize(X[i:i + bs]); xb = xb.flip(3) if flip else xb
            xb = xb.clone().requires_grad_(True)
            with ac(): z = net(xb).float()
            top = z.argmax(1)
            grad, = torch.autograd.grad(z.gather(1, top[:, None]).sum(), xb)
            sal = F.avg_pool2d(grad.abs().sum(1, keepdim=True).float(), 3, 1, 1).flatten(1)
            thr = torch.quantile(sal, 1 - keep, dim=1, keepdim=True)
            mask = (sal >= thr).view(-1, 1, 32, 32).to(xb.dtype)
            with torch.no_grad(), ac():
                lm = lm + net((xb.detach() * mask).contiguous(memory_format=torch.channels_last)).float()
            lo = lo + z.detach()
        out_o.append(lo / 2); out_m.append(lm / 2)
    return torch.cat(out_o), torch.cat(out_m)
KEEPS = [0.5, 0.7, 0.9]
ML = {k: {keep: masked_logits(X, keep) for keep in KEEPS} for k, X in DATA.items()}
def sal(split, p):
    keep, w = p
    lo, lm = ML[split][keep]
    return ((1 - w) * lo + w * lm).softmax(1)
GRID4 = [(k, w) for k in KEEPS for w in (0.1, 0.2, 0.3, 0.5)]
cv4, p4 = crossfit(sal, GRID4)
print(f"4. SALIENCY MASK: cross-fitted val {cv4:.4f} | setting chosen on all val keep={p4[0]}, w={p4[1]} | internal test {acc(sal('test', p4), yte_int):.4f}")
print(f"(done in {time.time() - t0:.0f}s)")

# ======================= decision on val only =======================
cur = acc(P0["val"], yva)
best_name, best_cv, best_fn = max([("gated TTA", cv3, lambda s: gated(s, thr3)), ("saliency mask", cv4, lambda s: sal(s, p4))], key=lambda r: r[1])
print(f"\nCURRENT val {cur:.4f} | best: {best_name}, cross-fitted val {best_cv:.4f}")
if best_cv > cur + 0.002:
    kp = best_fn("kaggle").argmax(1).cpu().numpy(); kb = P0["kaggle"].argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    tag = best_name.replace(" ", "_")
    drive_sub = os.path.join(RUN_DIR, f"submission_{tag}_cvval{best_cv:.4f}.csv"); sub.to_csv(drive_sub, index=False)
    os.makedirs(os.path.join(WORK_DIR, "gate_sal"), exist_ok=True); local_sub = os.path.join(WORK_DIR, "gate_sal", "submission.csv"); sub.to_csv(local_sub, index=False)
    print(f"DECISION: use {best_name}. saved {drive_sub}\nKaggle predictions that differ from the 0.915 submission: {(kp != kb).sum()} of {len(kp)}")
    if IN_COLAB:
        from google.colab import files; files.download(local_sub)
else:
    print("DECISION: keep the 0.915 submission (no cross-fitted VAL gain > 0.2 pt). No file written.")

CURRENT: val 0.9852 | internal test 0.9812   (expected 0.9852 / 0.9812)

3. GATED TTA: cross-fitted val 0.9848 | threshold chosen on all val 0.5 (13 of 2500 val images below it) | internal test 0.9816
4. SALIENCY MASK: cross-fitted val 0.9852 | setting chosen on all val keep=0.5, w=0.1 | internal test 0.9800
(done in 24s)

CURRENT val 0.9852 | best: saliency mask, cross-fitted val 0.9852
DECISION: keep the 0.915 submission (no cross-fitted VAL gain > 0.2 pt). No file written.


In [6]:
# ======================= Nearest class mean (prototypes from TRAINING images only) =======================
# Features = pooled output before the classifier (image + mirror averaged). Prototype = mean training feature per class.
# Prediction = closest prototype. Variant chosen on VAL (cross-fitted); internal test is the honest check.
Xtr_ncm = to_dev(X_full[sp["train"]]); ytr_ncm = torch.from_numpy(y_full[sp["train"]]).to(device)

@torch.no_grad()
def feats(X, bs=500):
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs])
        with ac():
            f = 0.5 * (net.features(xb).float() + net.features(xb.flip(3)).float())
        out.append(f.float())
    return torch.cat(out)

t0 = time.time()
Ftr = feats(Xtr_ncm)
FE = {"val": feats(Xva), "test": feats(Xte_int), "kaggle": feats(Xkaggle)}
mu = Ftr.mean(0, keepdim=True)                                         # global mean of training features (for centering)
PROTO = torch.stack([Ftr[ytr_ncm == k].mean(0) for k in range(10)])     # 10 class prototypes
print(f"features: train {tuple(Ftr.shape)} in {time.time() - t0:.0f}s")

def ncm(split, kind):
    f = FE[split]
    if kind == "cosine":          return F.normalize(f, dim=1) @ F.normalize(PROTO, dim=1).T
    if kind == "centered cosine": return F.normalize(f - mu, dim=1) @ F.normalize(PROTO - mu, dim=1).T
    return -torch.cdist(f, PROTO)                                      # "euclidean"

KINDS = ["cosine", "centered cosine", "euclidean"]
for k in KINDS:
    agree = (ncm("val", k).argmax(1) == P0["val"].argmax(1)).float().mean().item()
    print(f"  {k:16s}: val {acc(ncm('val', k), yva):.4f} | internal test {acc(ncm('test', k), yte_int):.4f} | agrees with current head on {agree:.1%} of val")
cv_ncm, k_ncm = crossfit(ncm, KINDS)
cur = acc(P0["val"], yva)
print(f"\nCURRENT val {cur:.4f} | internal test {acc(P0['test'], yte_int):.4f}")
print(f"NCM ({k_ncm}, chosen on val): cross-fitted val {cv_ncm:.4f} | internal test {acc(ncm('test', k_ncm), yte_int):.4f} "
      f"-> change {int(round((acc(ncm('test', k_ncm), yte_int) - acc(P0['test'], yte_int)) * len(yte_int))):+d} images")

if cv_ncm > cur + 0.002:
    kp = ncm("kaggle", k_ncm).argmax(1).cpu().numpy(); kb = P0["kaggle"].argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    drive_sub = os.path.join(RUN_DIR, f"submission_ncm_{k_ncm.replace(' ', '_')}_cvval{cv_ncm:.4f}.csv"); sub.to_csv(drive_sub, index=False)
    os.makedirs(os.path.join(WORK_DIR, "ncm"), exist_ok=True); local_sub = os.path.join(WORK_DIR, "ncm", "submission.csv"); sub.to_csv(local_sub, index=False)
    print(f"DECISION: use NCM. saved {drive_sub}\nKaggle predictions that differ from the 0.915 submission: {(kp != kb).sum()} of {len(kp)}")
    if IN_COLAB:
        from google.colab import files; files.download(local_sub)
else:
    print("DECISION: keep the 0.915 submission (no cross-fitted VAL gain > 0.2 pt). No file written.")

features: train (45000, 1040) in 6s
  cosine          : val 0.9816 | internal test 0.9792 | agrees with current head on 99.2% of val
  centered cosine : val 0.9832 | internal test 0.9792 | agrees with current head on 99.4% of val
  euclidean       : val 0.9800 | internal test 0.9792 | agrees with current head on 99.1% of val

CURRENT val 0.9852 | internal test 0.9812
NCM (centered cosine, chosen on val): cross-fitted val 0.9816 | internal test 0.9792 -> change -5 images
DECISION: keep the 0.915 submission (no cross-fitted VAL gain > 0.2 pt). No file written.


In [13]:
# ======================= Model soup: average the WEIGHTS of models fine-tuned from the 0.915 weights =======================
# One network (same architecture and params). Wortsman et al., ICML 2022. Only models that started from the 0.915 best.pt.
SOUP_RUNS = ["kd_sewrn_only_student_resnext29_slasd_400ep_v1",                       # the 0.915 model
             "kd_sewrn_only_student_resnext29_slasd_400ep_v1_ft100_lr0.02",          # +100 epochs fine-tune
             "kd_sewrn_only_student_resnext29_slasd_400ep_v1_tail50_noheavyaug"]      # refined-augmentation tail
SD = {}
for r in SOUP_RUNS:
    p = os.path.join(ROOT, "runs", r, "best.pt")
    if os.path.exists(p): SD[r] = torch.load(p, map_location=device, weights_only=False)["model"]
    else: print("missing (skipped):", p)
print("ingredients:", list(SD))

def soup_net(names):
    m = copy.deepcopy(net) if "copy" in globals() else None
    if m is None:
        import copy as _c; m = _c.deepcopy(net)
    avg = {}
    for k in SD[names[0]]:
        t = SD[names[0]][k]
        avg[k] = sum(SD[n][k].float() for n in names) / len(names) if t.is_floating_point() else t
    m.load_state_dict(avg); m.eval(); return m

@torch.no_grad()
def flip_probs(m, X, bs=500):
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs])
        with ac(): out.append((m(xb).float().softmax(1) + m(xb.flip(3)).float().softmax(1)) / 2)
    return torch.cat(out)

def evaluate(names):
    m = soup_net(names)
    return {s: flip_probs(m, X) for s, X in {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}.items()}

single = {r: evaluate([r]) for r in SD}
for r, p in single.items():
    print(f"  single {r[-40:]:40s}: val {acc(p['val'], yva):.4f} | internal test {acc(p['test'], yte_int):.4f}")

uniform = evaluate(list(SD))
print(f"\nUNIFORM SOUP ({len(SD)} models): val {acc(uniform['val'], yva):.4f} | internal test {acc(uniform['test'], yte_int):.4f}")

# greedy soup (Wortsman et al.): start from the best single model on val, add an ingredient only if val does not drop
order = sorted(SD, key=lambda r: acc(single[r]["val"], yva), reverse=True)
greedy = [order[0]]; g_val = acc(single[order[0]]["val"], yva)
for r in order[1:]:
    cand = evaluate(greedy + [r]); v = acc(cand["val"], yva)
    if v >= g_val: greedy.append(r); g_val = v
greedy_res = evaluate(greedy)
print(f"GREEDY SOUP {[g[-30:] for g in greedy]}: val {acc(greedy_res['val'], yva):.4f} (chosen on val, optimistic) | internal test {acc(greedy_res['test'], yte_int):.4f}")

cur = acc(P0["val"], yva)
print(f"\nCURRENT val {cur:.4f} | internal test {acc(P0['test'], yte_int):.4f}")
# decision uses the UNIFORM soup (no choices made on val, so its val score is not inflated)
if acc(uniform["val"], yva) > cur + 0.002:
    kp = uniform["kaggle"].argmax(1).cpu().numpy(); kb = P0["kaggle"].argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    drive_sub = os.path.join(RUN_DIR, f"submission_uniform_soup_{len(SD)}_val{acc(uniform['val'], yva):.4f}.csv"); sub.to_csv(drive_sub, index=False)
    torch.save({"model": soup_net(list(SD)).state_dict(), "cfg": CFG, "norm": best["norm"], "soup_of": list(SD)}, os.path.join(RUN_DIR, "best_uniform_soup.pt"))
    os.makedirs(os.path.join(WORK_DIR, "soup"), exist_ok=True); local_sub = os.path.join(WORK_DIR, "soup", "submission.csv"); sub.to_csv(local_sub, index=False)
    print(f"DECISION: use the uniform soup. saved {drive_sub} and best_uniform_soup.pt\nKaggle predictions that differ from the 0.915 submission: {(kp != kb).sum()} of {len(kp)}")
    if IN_COLAB:
        from google.colab import files; files.download(local_sub)
else:
    print("DECISION: keep the 0.915 submission (uniform soup does not beat it on VAL by > 0.2 pt). No file written.")

ingredients: ['kd_sewrn_only_student_resnext29_slasd_400ep_v1', 'kd_sewrn_only_student_resnext29_slasd_400ep_v1_ft100_lr0.02', 'kd_sewrn_only_student_resnext29_slasd_400ep_v1_tail50_noheavyaug']
  single rn_only_student_resnext29_slasd_400ep_v1: val 0.9852 | internal test 0.9812
  single nt_resnext29_slasd_400ep_v1_ft100_lr0.02: val 0.9876 | internal test 0.9828
  single snext29_slasd_400ep_v1_tail50_noheavyaug: val 0.9852 | internal test 0.9800

UNIFORM SOUP (3 models): val 0.9868 | internal test 0.9824
GREEDY SOUP ['29_slasd_400ep_v1_ft100_lr0.02']: val 0.9876 (chosen on val, optimistic) | internal test 0.9828

CURRENT val 0.9852 | internal test 0.9812
DECISION: keep the 0.915 submission (uniform soup does not beat it on VAL by > 0.2 pt). No file written.


In [14]:
kp = uniform["kaggle"].argmax(1).cpu().numpy(); kb = P0["kaggle"].argmax(1).cpu().numpy()
sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
drive_sub = os.path.join(RUN_DIR, "submission_uniform_soup3_val0.9868_test0.9824.csv"); sub.to_csv(drive_sub, index=False)
torch.save({"model": soup_net(list(SD)).state_dict(), "cfg": CFG, "norm": best["norm"], "soup_of": list(SD)}, os.path.join(RUN_DIR, "best_uniform_soup.pt"))
os.makedirs(os.path.join(WORK_DIR, "soup"), exist_ok=True); local_sub = os.path.join(WORK_DIR, "soup", "submission.csv"); sub.to_csv(local_sub, index=False)
print(f"saved {drive_sub}\nKaggle predictions that differ from the 0.915 submission: {(kp != kb).sum()} of {len(kp)}")
if IN_COLAB:
    from google.colab import files; files.download(local_sub)

saved /content/drive/MyDrive/cifar10_resnet_challenge/runs/kd_sewrn_only_student_resnext29_slasd_400ep_v1/submission_uniform_soup3_val0.9868_test0.9824.csv
Kaggle predictions that differ from the 0.915 submission: 20 of 2000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# ======================= WiSE-FT-style interpolation: (1 - a) * 0.915 weights + a * uniform soup weights =======================
# a = 0 -> the 0.915 model, a = 1 -> the uniform soup (your 0.917 submission). a chosen on VAL (cross-fitted). One network.
import copy
BASE = "kd_sewrn_only_student_resnext29_slasd_400ep_v1"
SOUP_W = {k: (sum(SD[n][k].float() for n in SD) / len(SD)) if SD[BASE][k].is_floating_point() else SD[BASE][k] for k in SD[BASE]}

def interp_net(a):
    m = copy.deepcopy(net)
    m.load_state_dict({k: ((1 - a) * SD[BASE][k].float() + a * SOUP_W[k]) if SD[BASE][k].is_floating_point() else SD[BASE][k]
                       for k in SD[BASE]})
    m.eval(); return m

ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]                         # > 1 extrapolates past the soup
t0 = time.time()
WI = {a: {s: flip_probs(interp_net(a), X) for s, X in {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}.items()} for a in ALPHAS}
print(f"computed in {time.time() - t0:.0f}s")
for a in ALPHAS:
    print(f"  a={a:<4}: val {acc(WI[a]['val'], yva):.4f} | internal test {acc(WI[a]['test'], yte_int):.4f}")

wi_score = lambda split, a: WI[split if False else a][split] if False else WI[a][split]
cv_wi, a_wi = crossfit(lambda split, a: WI[a][split], ALPHAS)
soup_val, soup_test = acc(uniform["val"], yva), acc(uniform["test"], yte_int)
print(f"\nSOUP (a=1, your 0.917 submission): val {soup_val:.4f} | internal test {soup_test:.4f}")
print(f"INTERPOLATION (a={a_wi}, chosen on val): cross-fitted val {cv_wi:.4f} | internal test {acc(WI[a_wi]['test'], yte_int):.4f} "
      f"-> change vs soup {int(round((acc(WI[a_wi]['test'], yte_int) - soup_test) * len(yte_int))):+d} images")

if a_wi != 1.0 and cv_wi > soup_val + 0.002:
    kp = WI[a_wi]["kaggle"].argmax(1).cpu().numpy(); kb = uniform["kaggle"].argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    drive_sub = os.path.join(RUN_DIR, f"submission_wise_a{a_wi}_cvval{cv_wi:.4f}.csv"); sub.to_csv(drive_sub, index=False)
    torch.save({"model": interp_net(a_wi).state_dict(), "cfg": CFG, "norm": best["norm"], "alpha": a_wi, "soup_of": list(SD)},
               os.path.join(RUN_DIR, f"best_wise_a{a_wi}.pt"))
    os.makedirs(os.path.join(WORK_DIR, "wise"), exist_ok=True); local_sub = os.path.join(WORK_DIR, "wise", "submission.csv"); sub.to_csv(local_sub, index=False)
    print(f"DECISION: use a={a_wi}. saved {drive_sub}\nKaggle predictions that differ from the soup submission: {(kp != kb).sum()} of {len(kp)}")
    if IN_COLAB:
        from google.colab import files; files.download(local_sub)
else:
    print("DECISION: keep the soup submission (no interpolation beats it on VAL by > 0.2 pt). No file written.")

computed in 6s
  a=0.0 : val 0.9852 | internal test 0.9812
  a=0.25: val 0.9860 | internal test 0.9816
  a=0.5 : val 0.9860 | internal test 0.9808
  a=0.75: val 0.9868 | internal test 0.9808
  a=1.0 : val 0.9868 | internal test 0.9824
  a=1.25: val 0.9872 | internal test 0.9824
  a=1.5 : val 0.9872 | internal test 0.9828

SOUP (a=1, your 0.917 submission): val 0.9868 | internal test 0.9824
INTERPOLATION (a=1.25, chosen on val): cross-fitted val 0.9864 | internal test 0.9824 -> change vs soup +0 images
DECISION: keep the soup submission (no interpolation beats it on VAL by > 0.2 pt). No file written.


In [16]:
# ======================= Mahalanobis (shared covariance) classifier on penultimate features =======================
# Class means and ONE covariance from the 45k TRAINING features; shrinkage toward the identity chosen on VAL (cross-fitted).
Xc = Ftr - PROTO[ytr_ncm]                                               # training features centred on their class mean
COV = (Xc.T @ Xc) / len(Xc)
d = COV.shape[0]; tr = COV.trace() / d

def maha(split, shrink):
    S = (1 - shrink) * COV + shrink * tr * torch.eye(d, device=COV.device)
    P = torch.linalg.inv(S)
    diff = FE[split][:, None, :] - PROTO[None, :, :]                     # (n, 10, d)
    return -torch.einsum("nkd,de,nke->nk", diff, P, diff)               # higher = closer

SHRINKS = [0.0, 0.01, 0.1, 0.5]
for s_ in SHRINKS:
    agree = (maha("val", s_).argmax(1) == P0["val"].argmax(1)).float().mean().item()
    print(f"  shrink {s_:<5}: val {acc(maha('val', s_), yva):.4f} | internal test {acc(maha('test', s_), yte_int):.4f} | agrees with current head {agree:.1%}")
cv_mh, s_mh = crossfit(maha, SHRINKS)
best_val, best_test = acc(uniform["val"], yva), acc(uniform["test"], yte_int)
print(f"\nSOUP (your 0.917): val {best_val:.4f} | internal test {best_test:.4f}   | 0.915 model: val {acc(P0['val'], yva):.4f} | internal test {acc(P0['test'], yte_int):.4f}")
print(f"MAHALANOBIS (shrink {s_mh}, chosen on val): cross-fitted val {cv_mh:.4f} | internal test {acc(maha('test', s_mh), yte_int):.4f}")
print("DECISION:", "beats the soup on val by > 0.2 pt -> worth writing a file" if cv_mh > best_val + 0.002 else "keep the soup submission")

  shrink 0.0  : val 0.9832 | internal test 0.9816 | agrees with current head 99.5%
  shrink 0.01 : val 0.9840 | internal test 0.9820 | agrees with current head 99.6%
  shrink 0.1  : val 0.9840 | internal test 0.9808 | agrees with current head 99.6%
  shrink 0.5  : val 0.9840 | internal test 0.9812 | agrees with current head 99.8%

SOUP (your 0.917): val 0.9868 | internal test 0.9824   | 0.915 model: val 0.9852 | internal test 0.9812
MAHALANOBIS (shrink 0.01, chosen on val): cross-fitted val 0.9828 | internal test 0.9820
DECISION: keep the soup submission


In [19]:
# ======================= Bigger soup from weights you ALREADY have (no training) =======================
# Extra ingredients from each run's last.pt: final-epoch raw weights ("model") and final EMA weights ("ema").
# All share the 0.915 starting point. Uniform + cross-fitted greedy soup, judged against the current 0.917 soup on VAL.
import copy
POOL = {f"best:{k[-30:]}": v for k, v in SD.items()}
for r in SD:
    p = os.path.join(ROOT, "runs", r, "last.pt")
    if not os.path.exists(p): print("no last.pt:", r); continue
    ck = torch.load(p, map_location=device, weights_only=False)
    POOL[f"raw:{r[-30:]}"] = ck["model"]
    if ck.get("ema"):
        ema = {k[len("module."):]: v for k, v in ck["ema"].items() if k.startswith("module.")}
        if ema: POOL[f"ema:{r[-30:]}"] = ema
print(f"{len(POOL)} ingredients:", list(POOL))

def make_net(names):
    m = copy.deepcopy(net); ref = POOL[names[0]]
    m.load_state_dict({k: (sum(POOL[n][k].float() for n in names) / len(names)) if ref[k].is_floating_point() else ref[k] for k in ref})
    m.eval(); return m
def probs_of(names, splits=("val",)):
    m = make_net(names); return {s: flip_probs(m, {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}[s]) for s in splits}

VALP = {n: probs_of([n], ("val", "test")) for n in POOL}
for n in POOL: print(f"  single {n:40s} val {acc(VALP[n]['val'], yva):.4f} | internal test {acc(VALP[n]['test'], yte_int):.4f}")

def greedy_on(ix):
    order = sorted(POOL, key=lambda n: acc(VALP[n]["val"][ix], yva[ix]), reverse=True)
    chosen = [order[0]]; best_a = acc(VALP[order[0]]["val"][ix], yva[ix])
    for n in order[1:]:
        a = acc(probs_of(chosen + [n])["val"][ix], yva[ix])
        if a >= best_a: chosen.append(n); best_a = a
    return chosen
cv_correct = 0
for a, b in FOLDS:
    g_ = greedy_on(a); cv_correct += (probs_of(g_)["val"][b].argmax(1) == yva[b]).sum().item()
cv_greedy = cv_correct / len(yva)
G = greedy_on(torch.arange(len(yva), device=device)); GR = probs_of(G, ("val", "test", "kaggle"))
U = probs_of(list(POOL), ("val", "test", "kaggle"))

cur_val, cur_test = acc(uniform["val"], yva), acc(uniform["test"], yte_int)
print(f"\nCURRENT 0.917 SOUP: val {cur_val:.4f} | internal test {cur_test:.4f}")
print(f"UNIFORM SOUP ({len(POOL)}): val {acc(U['val'], yva):.4f} | internal test {acc(U['test'], yte_int):.4f}")
print(f"GREEDY SOUP ({len(G)}): cross-fitted val {cv_greedy:.4f} | internal test {acc(GR['test'], yte_int):.4f}  {G}")

name, v, R, members = max([("uniform", acc(U["val"], yva), U, list(POOL)), ("greedy", cv_greedy, GR, G)], key=lambda c: c[1])
if v > cur_val + 0.002:
    kp = R["kaggle"].argmax(1).cpu().numpy(); kb = uniform["kaggle"].argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    tag = f"soup_{name}_{len(members)}_val{v:.4f}_test{acc(R['test'], yte_int):.4f}"
    drive_sub = os.path.join(RUN_DIR, f"submission_{tag}.csv"); sub.to_csv(drive_sub, index=False)
    torch.save({"model": make_net(members).state_dict(), "cfg": CFG, "norm": best["norm"], "soup_of": members}, os.path.join(RUN_DIR, f"best_{tag}.pt"))
    os.makedirs(os.path.join(WORK_DIR, "soup2"), exist_ok=True); local_sub = os.path.join(WORK_DIR, "soup2", "submission.csv"); sub.to_csv(local_sub, index=False)
    print(f"DECISION: use the {name} soup. saved {drive_sub}\nKaggle predictions that differ from the 0.917 soup: {(kp != kb).sum()} of {len(kp)}")
    if IN_COLAB:
        from google.colab import files; files.download(local_sub)
else:
    print("DECISION: keep the 0.917 soup (nothing beats it on VAL by > 0.2 pt). No file written.")

9 ingredients: ['best:udent_resnext29_slasd_400ep_v1', 'best:29_slasd_400ep_v1_ft100_lr0.02', 'best:asd_400ep_v1_tail50_noheavyaug', 'raw:udent_resnext29_slasd_400ep_v1', 'ema:udent_resnext29_slasd_400ep_v1', 'raw:29_slasd_400ep_v1_ft100_lr0.02', 'ema:29_slasd_400ep_v1_ft100_lr0.02', 'raw:asd_400ep_v1_tail50_noheavyaug', 'ema:asd_400ep_v1_tail50_noheavyaug']
  single best:udent_resnext29_slasd_400ep_v1      val 0.9852 | internal test 0.9812
  single best:29_slasd_400ep_v1_ft100_lr0.02      val 0.9876 | internal test 0.9828
  single best:asd_400ep_v1_tail50_noheavyaug      val 0.9852 | internal test 0.9800
  single raw:udent_resnext29_slasd_400ep_v1       val 0.9848 | internal test 0.9824
  single ema:udent_resnext29_slasd_400ep_v1       val 0.9852 | internal test 0.9812
  single raw:29_slasd_400ep_v1_ft100_lr0.02       val 0.9876 | internal test 0.9820
  single ema:29_slasd_400ep_v1_ft100_lr0.02       val 0.9868 | internal test 0.9820
  single raw:asd_400ep_v1_tail50_noheavyaug       v

In [20]:
# ======================= ONE SOUP INGREDIENT: short fine-tune from the 0.915 weights (A100: ~20-30 min) =======================
# Same loss as the 0.915 recipe: SLA joint CE on 4 rotations + 10-class head = 0.9 KD (SE-WRN teacher) + 0.1 SLA self-distillation.
# Same augmentation: TrivialAugment, crop+flip, cutout 16, CutMix on 50% of batches. Plain SGD-Nesterov (no ASAM, faster), EMA 0.999.
# Saves the EMA weights every 5 epochs to runs/<BASE>_<ING_NAME>/ and into EXTRA_ING for the soup.
ING_NAME, ING_SEED, ING_LR, ING_EPOCHS = "ing1", 21, 0.01, 30          # change per run: ing2/22/0.02, ing3/23/0.005 ...
import copy, math
from torchvision.transforms import v2 as T
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
torch.manual_seed(ING_SEED); np.random.seed(ING_SEED)
BASE = "kd_sewrn_only_student_resnext29_slasd_400ep_v1"
ING_DIR = os.path.join(ROOT, "runs", f"{BASE}_{ING_NAME}"); os.makedirs(ING_DIR, exist_ok=True)

# data (TRAINING split only) and teacher
Xtr_i = to_dev(X_full[sp["train"]]); ytr_i = torch.from_numpy(y_full[sp["train"]]).to(device)
tck = torch.load(os.path.join(ROOT, "runs", "teacher_sewrn28_10_sla250_v1", "best.pt"), map_location=device, weights_only=False)
teacher = build_model(tck["cfg"]).to(device).to(memory_format=torch.channels_last); teacher.load_state_dict(tck["model"]); teacher.eval()
T_CAL = 0.96                                                             # the teacher's calibrated temperature (your 9b output)

TA = T.TrivialAugmentWide()
def augment(xu8):
    xu8 = torch.cat([TA(c) for c in xu8.split(16)])
    x = normalize(xu8); n, ar = len(x), torch.arange(32, device=device)
    xp = F.pad(x, (4, 4, 4, 4), mode="reflect")
    dy, dx = torch.randint(0, 9, (n,), device=device), torch.randint(0, 9, (n,), device=device)
    x = xp[torch.arange(n, device=device)[:, None, None, None], torch.arange(3, device=device)[None, :, None, None],
           (dy[:, None] + ar)[:, None, :, None], (dx[:, None] + ar)[:, None, None, :]]
    x = torch.where((torch.rand(n, device=device) < 0.5)[:, None, None, None], x.flip(3), x)
    cy, cx = torch.randint(0, 32, (n,), device=device), torch.randint(0, 32, (n,), device=device)
    hole = ((ar[None, :, None] - cy[:, None, None]).abs() < 8) & ((ar[None, None, :] - cx[:, None, None]).abs() < 8)
    return (x * (~hole)[:, None].float()).contiguous(memory_format=torch.channels_last)
def cutmix(x, y):
    lam = np.random.beta(1.0, 1.0); idx = torch.randperm(len(x), device=device)
    r = math.sqrt(1 - lam); ch = cw = int(32 * r); cy, cx = np.random.randint(32), np.random.randint(32)
    y1, y2, x1, x2 = max(cy - ch // 2, 0), min(cy + ch // 2, 32), max(cx - cw // 2, 0), min(cx + cw // 2, 32)
    x = x.clone(); x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    return x, y, y[idx], 1 - (y2 - y1) * (x2 - x1) / 1024

# student = the 0.915 weights; optimiser, cosine LR with 1-epoch warm-up, EMA
model = copy.deepcopy(net).train()
decay = [p for p in model.parameters() if p.ndim > 1]; no_decay = [p for p in model.parameters() if p.ndim <= 1]
opt = torch.optim.SGD([{"params": decay, "weight_decay": 5e-4}, {"params": no_decay, "weight_decay": 0.0}], lr=ING_LR, momentum=0.9, nesterov=True)
BS = 256; steps = math.ceil(len(Xtr_i) / BS); total = ING_EPOCHS * steps
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / steps) * 0.5 * (1 + math.cos(math.pi * min(s, total) / total)))
ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999), use_buffers=True)
rot = torch.arange(4, device=device)
EXTRA_ING = globals().get("EXTRA_ING", {})
print(f"{ING_NAME}: start val {acc(flip_probs(net, Xva), yva):.4f} | internal {acc(flip_probs(net, Xte_int), yte_int):.4f}  (expected 0.9852 / 0.9812)")

for ep in range(ING_EPOCHS):
    t0 = time.time(); model.train(); perm = torch.randperm(len(Xtr_i), device=device); tot = 0.0
    for i in range(0, len(Xtr_i), BS):
        bi = perm[i:i + BS]; x = augment(Xtr_i[bi]); y = ytr_i[bi]; ya, yb, lam = y, y, 1.0
        if np.random.rand() < 0.5: x, ya, yb, lam = cutmix(x, y)
        B = len(x)
        with torch.no_grad(), torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            t_prob = (teacher(x).float() / T_CAL).softmax(1)
        x4 = torch.cat([torch.rot90(x, k, (2, 3)) for k in range(4)]).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            sd_lo, j_lo = forward_both(model, x4)
        sd_lo, j_lo = sd_lo.float(), j_lo.float(); r4 = rot.repeat_interleave(B)
        lj = lam * F.cross_entropy(j_lo, ya.repeat(4) * 4 + r4) + (1 - lam) * F.cross_entropy(j_lo, yb.repeat(4) * 4 + r4)
        agg = torch.stack([j_lo.view(4, B, 10, 4)[k, :, :, k] for k in range(4)]).mean(0); sd0 = sd_lo[:B]
        self_d = F.kl_div(F.log_softmax(sd0, 1), F.softmax(agg.detach(), 1), reduction="batchmean")
        kd = F.kl_div(F.log_softmax(sd0, 1), t_prob, reduction="batchmean")
        loss = lj + (0.9 * kd + 0.1 * self_d)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step(); ema.update_parameters(model)
        tot += loss.item() * B
    msg = f"{ING_NAME} ep {ep+1:2d}/{ING_EPOCHS} | loss {tot / len(Xtr_i):.4f} | lr {sched.get_last_lr()[0]:.4f} | {time.time() - t0:.0f}s"
    if (ep + 1) % 5 == 0:
        sd_ = {k: v.detach().clone() for k, v in ema.module.state_dict().items()}
        torch.save({"model": sd_, "epoch": ep + 1}, os.path.join(ING_DIR, f"snap_e{ep + 1:03d}.pt"))
        EXTRA_ING[f"{ING_NAME}_e{ep + 1:03d}"] = sd_
        m_ = copy.deepcopy(net); m_.load_state_dict(sd_); m_.eval()
        msg += f" | EMA val {acc(flip_probs(m_, Xva), yva):.4f} | internal {acc(flip_probs(m_, Xte_int), yte_int):.4f} | saved snapshot"
    print(msg, flush=True)
print(f"{ING_NAME} done: {len([k for k in EXTRA_ING if k.startswith(ING_NAME)])} snapshots in EXTRA_ING and {ING_DIR}")

ing1: start val 0.9852 | internal 0.9812  (expected 0.9852 / 0.9812)
ing1 ep  1/30 | loss 0.9574 | lr 0.0100 | 40s
ing1 ep  2/30 | loss 1.0070 | lr 0.0099 | 39s
ing1 ep  3/30 | loss 1.0279 | lr 0.0098 | 39s
ing1 ep  4/30 | loss 1.0874 | lr 0.0096 | 39s
ing1 ep  5/30 | loss 1.0506 | lr 0.0093 | 39s | EMA val 0.9832 | internal 0.9820 | saved snapshot
ing1 ep  6/30 | loss 1.0414 | lr 0.0090 | 39s
ing1 ep  7/30 | loss 1.0446 | lr 0.0087 | 39s
ing1 ep  8/30 | loss 1.0912 | lr 0.0083 | 39s
ing1 ep  9/30 | loss 1.0051 | lr 0.0079 | 39s
ing1 ep 10/30 | loss 1.0342 | lr 0.0075 | 39s | EMA val 0.9836 | internal 0.9820 | saved snapshot
ing1 ep 11/30 | loss 1.0539 | lr 0.0070 | 39s
ing1 ep 12/30 | loss 1.0769 | lr 0.0065 | 39s
ing1 ep 13/30 | loss 0.9953 | lr 0.0060 | 39s
ing1 ep 14/30 | loss 0.9663 | lr 0.0055 | 39s
ing1 ep 15/30 | loss 0.9625 | lr 0.0050 | 39s | EMA val 0.9848 | internal 0.9820 | saved snapshot
ing1 ep 16/30 | loss 0.9869 | lr 0.0045 | 39s
ing1 ep 17/30 | loss 0.9638 | lr 0.0040

In [21]:
# ======================= SOUP over everything fine-tuned from the 0.915 weights =======================
import copy, glob
BASE = "kd_sewrn_only_student_resnext29_slasd_400ep_v1"
POOL = {f"best:{k[-30:]}": v for k, v in SD.items()}                     # the 3 current ingredients (0.917 soup)
for r in SD:                                                             # final raw + final EMA weights from last.pt
    p = os.path.join(ROOT, "runs", r, "last.pt")
    if not os.path.exists(p): continue
    ck = torch.load(p, map_location=device, weights_only=False)
    POOL[f"raw:{r[-30:]}"] = ck["model"]
    if ck.get("ema"):
        e = {k[len("module."):]: v for k, v in ck["ema"].items() if k.startswith("module.")}
        if e: POOL[f"ema:{r[-30:]}"] = e
for p in sorted(glob.glob(os.path.join(ROOT, "runs", BASE + "_ing*", "snap_e*.pt"))):   # new ingredients (any GPU)
    POOL["snap:" + os.path.relpath(p, os.path.join(ROOT, "runs"))[len(BASE) + 1:]] = torch.load(p, map_location=device, weights_only=False)["model"]
print(f"{len(POOL)} ingredients")

def make_net(names):
    m = copy.deepcopy(net); ref = POOL[names[0]]
    m.load_state_dict({k: (sum(POOL[n][k].float() for n in names) / len(names)) if ref[k].is_floating_point() else ref[k] for k in ref})
    m.eval(); return m
def probs_of(names, splits=("val",)):
    m = make_net(names); return {s: flip_probs(m, {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}[s]) for s in splits}

t0 = time.time()
VALP = {n: probs_of([n], ("val", "test")) for n in POOL}
for n in POOL: print(f"  {n:45s} val {acc(VALP[n]['val'], yva):.4f} | internal test {acc(VALP[n]['test'], yte_int):.4f}")

def greedy_on(ix):
    order = sorted(POOL, key=lambda n: acc(VALP[n]["val"][ix], yva[ix]), reverse=True)
    chosen = [order[0]]; best_a = acc(VALP[order[0]]["val"][ix], yva[ix])
    for n in order[1:]:
        a = acc(probs_of(chosen + [n])["val"][ix], yva[ix])
        if a >= best_a: chosen.append(n); best_a = a
    return chosen
cv_correct = 0
for a, b in FOLDS:
    g_ = greedy_on(a); cv_correct += (probs_of(g_)["val"][b].argmax(1) == yva[b]).sum().item()
cv_greedy = cv_correct / len(yva)
G = greedy_on(torch.arange(len(yva), device=device)); GR = probs_of(G, ("val", "test", "kaggle"))
U = probs_of(list(POOL), ("val", "test", "kaggle"))
print(f"(done in {time.time() - t0:.0f}s)")

cur_val, cur_test = acc(uniform["val"], yva), acc(uniform["test"], yte_int)
print(f"\nCURRENT 0.917 SOUP: val {cur_val:.4f} | internal test {cur_test:.4f}   (expected 0.9868 / 0.9824)")
print(f"UNIFORM SOUP ({len(POOL)}): val {acc(U['val'], yva):.4f} | internal test {acc(U['test'], yte_int):.4f}")
print(f"GREEDY SOUP ({len(G)}): cross-fitted val {cv_greedy:.4f} | internal test {acc(GR['test'], yte_int):.4f}")
print("  greedy ingredients:", G)

name, v, R, members = max([("uniform", acc(U["val"], yva), U, list(POOL)), ("greedy", cv_greedy, GR, G)], key=lambda c: c[1])
if v > cur_val + 0.002:
    kp = R["kaggle"].argmax(1).cpu().numpy(); kb = uniform["kaggle"].argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    tag = f"soup_{name}_{len(members)}_val{v:.4f}_test{acc(R['test'], yte_int):.4f}"
    drive_sub = os.path.join(RUN_DIR, f"submission_{tag}.csv"); sub.to_csv(drive_sub, index=False)
    torch.save({"model": make_net(members).state_dict(), "cfg": CFG, "norm": best["norm"], "soup_of": members}, os.path.join(RUN_DIR, f"best_{tag}.pt"))
    os.makedirs(os.path.join(WORK_DIR, "soup2"), exist_ok=True); local_sub = os.path.join(WORK_DIR, "soup2", "submission.csv"); sub.to_csv(local_sub, index=False)
    print(f"DECISION: use the {name} soup. saved {drive_sub}\nKaggle predictions that differ from the 0.917 soup: {(kp != kb).sum()} of {len(kp)}")
    if IN_COLAB:
        from google.colab import files; files.download(local_sub)
else:
    print("DECISION: keep the 0.917 soup (nothing beats it on VAL by > 0.2 pt). No file written.")

15 ingredients
  best:udent_resnext29_slasd_400ep_v1           val 0.9852 | internal test 0.9812
  best:29_slasd_400ep_v1_ft100_lr0.02           val 0.9876 | internal test 0.9828
  best:asd_400ep_v1_tail50_noheavyaug           val 0.9852 | internal test 0.9800
  raw:udent_resnext29_slasd_400ep_v1            val 0.9848 | internal test 0.9824
  ema:udent_resnext29_slasd_400ep_v1            val 0.9852 | internal test 0.9812
  raw:29_slasd_400ep_v1_ft100_lr0.02            val 0.9876 | internal test 0.9820
  ema:29_slasd_400ep_v1_ft100_lr0.02            val 0.9868 | internal test 0.9820
  raw:asd_400ep_v1_tail50_noheavyaug            val 0.9828 | internal test 0.9800
  ema:asd_400ep_v1_tail50_noheavyaug            val 0.9832 | internal test 0.9804
  snap:ing1/snap_e005.pt                        val 0.9832 | internal test 0.9820
  snap:ing1/snap_e010.pt                        val 0.9836 | internal test 0.9820
  snap:ing1/snap_e015.pt                        val 0.9848 | internal test 0.9820
 

In [22]:
# ======================= Principal-subspace projection of penultimate features before the trained linear head =======================
# Subspace = top-K right singular vectors of the CENTRED 45k TRAINING features (0.915 model). K chosen on VAL (cross-fitted).
mu_tr = Ftr.mean(0, keepdim=True)
_, Svals, Vh = torch.linalg.svd(Ftr - mu_tr, full_matrices=False)       # Vh: (d, d) rows = principal directions
energy = (Svals ** 2).cumsum(0) / (Svals ** 2).sum()
print("variance captured by top 10 / 32 / 128 directions:", [round(energy[k - 1].item(), 4) for k in (10, 32, 128)])

def project(split, K):
    if K is None: return net.fc(FE[split])                               # no projection (reference)
    V = Vh[:K].T                                                         # (d, K)
    f = (FE[split] - mu_tr) @ V @ V.T + mu_tr
    return net.fc(f)

KS = [None, 9, 16, 32, 64, 128, 256]
for K in KS:
    print(f"  K={str(K):5s}: val {acc(project('val', K), yva):.4f} | internal test {acc(project('test', K), yte_int):.4f}")
cv_pj, K_pj = crossfit(project, KS)
print(f"\n0.915 model (flip probs): val {acc(P0['val'], yva):.4f} | internal test {acc(P0['test'], yte_int):.4f}"
      f"   | 0.917 soup: val {acc(uniform['val'], yva):.4f} | internal test {acc(uniform['test'], yte_int):.4f}")
print(f"PROJECTION (K={K_pj}, chosen on val): cross-fitted val {cv_pj:.4f} | internal test {acc(project('test', K_pj), yte_int):.4f}")
print("DECISION:", "beats the 0.915 model on val by > 0.2 pt (then re-run on the soup's features)" if cv_pj > acc(P0['val'], yva) + 0.002 else "no gain: keep the soup submission")

variance captured by top 10 / 32 / 128 directions: [0.8036, 0.9215, 0.9768]
  K=None : val 0.9852 | internal test 0.9812
  K=9    : val 0.9728 | internal test 0.9756
  K=16   : val 0.9820 | internal test 0.9804
  K=32   : val 0.9844 | internal test 0.9808
  K=64   : val 0.9852 | internal test 0.9808
  K=128  : val 0.9852 | internal test 0.9820
  K=256  : val 0.9856 | internal test 0.9816

0.915 model (flip probs): val 0.9852 | internal test 0.9812   | 0.917 soup: val 0.9868 | internal test 0.9824
PROJECTION (K=256, chosen on val): cross-fitted val 0.9848 | internal test 0.9816
DECISION: no gain: keep the soup submission


In [23]:
# ======================= Softmax + k-NN blend (k-NN pool = 45k TRAINING features of the 0.915 model) =======================
# P = a * P_softmax + (1 - a) * P_kNN; P_kNN from the k most cosine-similar training images, weighted by exp(sim / 0.07).
Fn_tr = F.normalize(Ftr, dim=1)
Ytr_1h = F.one_hot(ytr_ncm, 10).float()
def knn_probs(split, k, tau=0.07, bs=500):
    out = []
    f = F.normalize(FE[split], dim=1)
    for i in range(0, len(f), bs):
        sim = f[i:i + bs] @ Fn_tr.T                                      # (b, 45000)
        s, j = sim.topk(k, dim=1)
        w = (s / tau).exp()
        out.append((w[..., None] * Ytr_1h[j]).sum(1) / w.sum(1, keepdim=True))
    return torch.cat(out)
KNN = {(s, k): knn_probs(s, k) for s in ("val", "test", "kaggle") for k in (10, 50, 200)}

def blend(split, p):
    k, a = p
    return a * P0[split] + (1 - a) * KNN[(split, k)]
GRID_KNN = [(k, a) for k in (10, 50, 200) for a in (0.0, 0.5, 0.75, 0.9)]   # a = 0 -> k-NN alone
for g in GRID_KNN:
    print(f"  k={g[0]:<4} a={g[1]:<5}: val {acc(blend('val', g), yva):.4f} | internal test {acc(blend('test', g), yte_int):.4f}")
cv_kn, g_kn = crossfit(blend, GRID_KNN)
print(f"\n0.915 model: val {acc(P0['val'], yva):.4f} | internal test {acc(P0['test'], yte_int):.4f}"
      f"   | 0.917 soup: val {acc(uniform['val'], yva):.4f} | internal test {acc(uniform['test'], yte_int):.4f}")
print(f"SOFTMAX+kNN (k={g_kn[0]}, a={g_kn[1]}, chosen on val): cross-fitted val {cv_kn:.4f} | internal test {acc(blend('test', g_kn), yte_int):.4f}")
print("DECISION:", "beats the 0.915 model on val by > 0.2 pt (then worth redoing on the soup's features)" if cv_kn > acc(P0['val'], yva) + 0.002 else "no gain: keep the soup submission")

  k=10   a=0.0  : val 0.9856 | internal test 0.9808
  k=10   a=0.5  : val 0.9856 | internal test 0.9812
  k=10   a=0.75 : val 0.9848 | internal test 0.9824
  k=10   a=0.9  : val 0.9852 | internal test 0.9808
  k=50   a=0.0  : val 0.9856 | internal test 0.9796
  k=50   a=0.5  : val 0.9844 | internal test 0.9796
  k=50   a=0.75 : val 0.9848 | internal test 0.9808
  k=50   a=0.9  : val 0.9848 | internal test 0.9812
  k=200  a=0.0  : val 0.9844 | internal test 0.9796
  k=200  a=0.5  : val 0.9840 | internal test 0.9812
  k=200  a=0.75 : val 0.9844 | internal test 0.9820
  k=200  a=0.9  : val 0.9848 | internal test 0.9812

0.915 model: val 0.9852 | internal test 0.9812   | 0.917 soup: val 0.9868 | internal test 0.9824
SOFTMAX+kNN (k=10, a=0.0, chosen on val): cross-fitted val 0.9852 | internal test 0.9808
DECISION: no gain: keep the soup submission


In [24]:
# ======================= Inter-class weight-distance scaling of the logits =======================
# D = cosine distance between the 10 rows of the final layer; s_i = mean distance of class i to the other 9 classes.
# scale_i = (mean(s) / s_i) ** g  -> g > 0 boosts classes that sit CLOSE to others, g < 0 boosts isolated classes, g = 0 = no change.
# Logits are shifted per image to be >= 0 first (a per-image shift never changes the argmax), so the multiplier acts the same way
# on every class. g chosen on VAL (cross-fitted).
W = net.fc.weight.detach().float()
Cs = F.normalize(W, dim=1) @ F.normalize(W, dim=1).T
D = 1 - Cs
s = (D.sum(1) - D.diag()) / 9                                            # mean distance to the other classes
print("mean cosine distance per class:", dict(zip(["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"],
                                                  [round(v, 3) for v in s.tolist()])))

def wdist(split, g):
    z = net.fc(FE[split])                                                # logits from the (image + mirror)-averaged features
    z = z - z.min(1, keepdim=True).values                                # >= 0, same argmax
    return z * ((s.mean() / s) ** g)[None, :]

GS = [-1.0, -0.5, -0.25, 0.0, 0.25, 0.5, 1.0]
for g in GS:
    flips = (wdist("val", g).argmax(1) != wdist("val", 0.0).argmax(1)).sum().item()
    print(f"  g={g:<5}: val {acc(wdist('val', g), yva):.4f} | internal test {acc(wdist('test', g), yte_int):.4f} | val predictions changed: {flips}")
cv_wd, g_wd = crossfit(wdist, GS)
print(f"\n0.915 model: val {acc(P0['val'], yva):.4f} | internal test {acc(P0['test'], yte_int):.4f}"
      f"   | 0.917 soup: val {acc(uniform['val'], yva):.4f} | internal test {acc(uniform['test'], yte_int):.4f}")
print(f"WEIGHT-DISTANCE SCALING (g={g_wd}, chosen on val): cross-fitted val {cv_wd:.4f} | internal test {acc(wdist('test', g_wd), yte_int):.4f}")
print("DECISION:", "beats the 0.915 model on val by > 0.2 pt (then worth redoing on the soup)" if cv_wd > acc(P0['val'], yva) + 0.002 else "no gain: keep the soup submission")

mean cosine distance per class: {'plane': 1.112, 'car': 1.105, 'bird': 1.114, 'cat': 1.111, 'deer': 1.114, 'dog': 1.11, 'frog': 1.109, 'horse': 1.116, 'ship': 1.111, 'truck': 1.108}
  g=-1.0 : val 0.9852 | internal test 0.9812 | val predictions changed: 0
  g=-0.5 : val 0.9852 | internal test 0.9812 | val predictions changed: 0
  g=-0.25: val 0.9852 | internal test 0.9812 | val predictions changed: 0
  g=0.0  : val 0.9852 | internal test 0.9812 | val predictions changed: 0
  g=0.25 : val 0.9852 | internal test 0.9812 | val predictions changed: 0
  g=0.5  : val 0.9852 | internal test 0.9812 | val predictions changed: 0
  g=1.0  : val 0.9848 | internal test 0.9812 | val predictions changed: 1

0.915 model: val 0.9852 | internal test 0.9812   | 0.917 soup: val 0.9868 | internal test 0.9824
WEIGHT-DISTANCE SCALING (g=-1.0, chosen on val): cross-fitted val 0.9852 | internal test 0.9812
DECISION: no gain: keep the soup submission


In [25]:
# ======================= Post-hoc ZCA whitening of penultimate features, then the ORIGINAL linear head =======================
# Covariance from the 45k TRAINING features (never the test set). ZCA = U diag(((lam + eps) / (mean_lam + eps))^-1/2) U^T,
# applied around the training mean: f' = ZCA (f - mu) + mu. eps = infinity -> identity (= the original model).
# eps (shrinkage, relative to the mean eigenvalue) chosen on VAL (cross-fitted).
mu_z = Ftr.mean(0, keepdim=True)
Cov_z = ((Ftr - mu_z).T @ (Ftr - mu_z)) / len(Ftr)
lam, U = torch.linalg.eigh(Cov_z)
lam = lam.clamp_min(0); lam_mean = lam.mean()

def zca(split, eps_rel):
    if eps_rel is None: return net.fc(FE[split])                         # no whitening (reference)
    eps = eps_rel * lam_mean
    Wz = U @ torch.diag(((lam + eps) / (lam_mean + eps)) ** -0.5) @ U.T
    return net.fc((FE[split] - mu_z) @ Wz.T + mu_z)

EPS = [None, 10.0, 1.0, 0.1, 0.01, 0.001]
for e in EPS:
    flips = (zca("val", e).argmax(1) != zca("val", None).argmax(1)).sum().item()
    print(f"  eps={str(e):6s}: val {acc(zca('val', e), yva):.4f} | internal test {acc(zca('test', e), yte_int):.4f} | val predictions changed: {flips}")
cv_z, e_z = crossfit(zca, EPS)
print(f"\n0.915 model: val {acc(P0['val'], yva):.4f} | internal test {acc(P0['test'], yte_int):.4f}"
      f"   | 0.917 soup: val {acc(uniform['val'], yva):.4f} | internal test {acc(uniform['test'], yte_int):.4f}")
print(f"ZCA (eps={e_z}, chosen on val): cross-fitted val {cv_z:.4f} | internal test {acc(zca('test', e_z), yte_int):.4f}")
print("DECISION:", "beats the 0.915 model on val by > 0.2 pt (then worth redoing on the soup)" if cv_z > acc(P0['val'], yva) + 0.002 else "no gain: keep the soup submission")

  eps=None  : val 0.9852 | internal test 0.9812 | val predictions changed: 0
  eps=10.0  : val 0.9832 | internal test 0.9812 | val predictions changed: 10
  eps=1.0   : val 0.9420 | internal test 0.9356 | val predictions changed: 120
  eps=0.1   : val 0.8708 | internal test 0.8512 | val predictions changed: 298
  eps=0.01  : val 0.8280 | internal test 0.8132 | val predictions changed: 405
  eps=0.001 : val 0.8164 | internal test 0.8060 | val predictions changed: 437

0.915 model: val 0.9852 | internal test 0.9812   | 0.917 soup: val 0.9868 | internal test 0.9824
ZCA (eps=None, chosen on val): cross-fitted val 0.9852 | internal test 0.9812
DECISION: no gain: keep the soup submission


In [26]:
# ======================= "Kernel despiking" in feature space: f' = f * exp(-g * ||f - m||^2 / median_dist^2) =======================
# m = coordinate-wise median of the 45k TRAINING features. Distances normalised by the median training distance so g is unit-free.
m_med = Ftr.median(0, keepdim=True).values
d_tr = ((Ftr - m_med) ** 2).sum(1)
d0 = d_tr.median()
def despike(split, g):
    f = FE[split]
    c = torch.exp(-g * ((f - m_med) ** 2).sum(1, keepdim=True) / d0)
    return net.fc(f * c)
GSD = [0.0, 0.1, 0.5, 1.0, 2.0]
for g in GSD:
    flips = (despike("val", g).argmax(1) != despike("val", 0.0).argmax(1)).sum().item()
    print(f"  g={g:<4}: val {acc(despike('val', g), yva):.4f} | internal test {acc(despike('test', g), yte_int):.4f} | val predictions changed: {flips}")
cv_d, g_d = crossfit(despike, GSD)
print(f"\n0.915 model: val {acc(P0['val'], yva):.4f} | internal test {acc(P0['test'], yte_int):.4f}   | 0.917 soup: val {acc(uniform['val'], yva):.4f} | internal test {acc(uniform['test'], yte_int):.4f}")
print(f"DESPIKE (g={g_d}, chosen on val): cross-fitted val {cv_d:.4f} | internal test {acc(despike('test', g_d), yte_int):.4f}")
print("DECISION:", "beats the 0.915 model on val by > 0.2 pt" if cv_d > acc(P0['val'], yva) + 0.002 else "no gain: keep the soup submission")

  g=0.0 : val 0.9852 | internal test 0.9812 | val predictions changed: 0
  g=0.1 : val 0.9852 | internal test 0.9812 | val predictions changed: 0
  g=0.5 : val 0.9856 | internal test 0.9812 | val predictions changed: 1
  g=1.0 : val 0.9852 | internal test 0.9808 | val predictions changed: 2
  g=2.0 : val 0.9720 | internal test 0.9640 | val predictions changed: 45

0.915 model: val 0.9852 | internal test 0.9812   | 0.917 soup: val 0.9868 | internal test 0.9824
DESPIKE (g=0.5, chosen on val): cross-fitted val 0.9852 | internal test 0.9812
DECISION: no gain: keep the soup submission


In [27]:
# ======================= Intra-class variance normalisation, two runnable readings (stats from 45k TRAINING features) =======================
cls_mean = torch.stack([Ftr[ytr_ncm == k].mean(0) for k in range(10)])
cls_std  = torch.stack([Ftr[ytr_ncm == k].std(0) for k in range(10)]).clamp_min(1e-4)      # (10, d)
pooled_std = (Ftr - cls_mean[ytr_ncm]).std(0).clamp_min(1e-4)                            # (d,)
mu_f = Ftr.mean(0, keepdim=True)

def icvn_a(split, p):     # (a) divide each dimension by pooled within-class std ** p (around the training mean), original head
    s = (pooled_std / pooled_std.mean()) ** p
    return net.fc((FE[split] - mu_f) / s + mu_f)

def icvn_b(split, p):     # (b) per candidate class k: diagonal Gaussian distance to class mean k with class-k std ** p
    f = FE[split]
    z = (f[:, None, :] - cls_mean[None]) / (cls_std[None] ** p)
    return -(z ** 2).sum(2) - (2 * p * cls_std.log()).sum(1)[None]      # log-likelihood up to a constant

for nm, fn in (("a: pooled std + original head", icvn_a), ("b: per-class diagonal Gaussian", icvn_b)):
    print(nm)
    for p in (0.0, 0.25, 0.5, 1.0):
        print(f"   p={p:<4}: val {acc(fn('val', p), yva):.4f} | internal test {acc(fn('test', p), yte_int):.4f}")
cv_a, p_a = crossfit(icvn_a, [0.0, 0.25, 0.5, 1.0]); cv_b, p_b = crossfit(icvn_b, [0.0, 0.25, 0.5, 1.0])
print(f"\n0.915 model: val {acc(P0['val'], yva):.4f} | internal test {acc(P0['test'], yte_int):.4f}   | 0.917 soup: val {acc(uniform['val'], yva):.4f} | internal test {acc(uniform['test'], yte_int):.4f}")
print(f"(a) p={p_a}: cross-fitted val {cv_a:.4f} | internal test {acc(icvn_a('test', p_a), yte_int):.4f}")
print(f"(b) p={p_b}: cross-fitted val {cv_b:.4f} | internal test {acc(icvn_b('test', p_b), yte_int):.4f}")
bestv = max(cv_a, cv_b)
print("DECISION:", "beats the 0.915 model on val by > 0.2 pt" if bestv > acc(P0['val'], yva) + 0.002 else "no gain: keep the soup submission")

a: pooled std + original head
   p=0.0 : val 0.9852 | internal test 0.9812
   p=0.25: val 0.9852 | internal test 0.9812
   p=0.5 : val 0.9852 | internal test 0.9812
   p=1.0 : val 0.9848 | internal test 0.9812
b: per-class diagonal Gaussian
   p=0.0 : val 0.9800 | internal test 0.9792
   p=0.25: val 0.9776 | internal test 0.9788
   p=0.5 : val 0.9836 | internal test 0.9800
   p=1.0 : val 0.9752 | internal test 0.9728

0.915 model: val 0.9852 | internal test 0.9812   | 0.917 soup: val 0.9868 | internal test 0.9824
(a) p=0.0: cross-fitted val 0.9848 | internal test 0.9812
(b) p=0.5: cross-fitted val 0.9800 | internal test 0.9800
DECISION: no gain: keep the soup submission


In [28]:
# CutMix whose partner image is drawn more often from WEAK classes (weights from the 0.915 model's per-class VAL accuracy)
_pv = P0["val"].argmax(1)
_cls_acc = torch.stack([(_pv[yva == k] == k).float().mean() for k in range(10)])
_w_cls = (1 - _cls_acc + 0.01) / (1 - _cls_acc + 0.01).mean()          # weak classes get weight > 1
print("per-class val accuracy:", [round(v, 3) for v in _cls_acc.tolist()], "\npartner weights:", [round(v, 2) for v in _w_cls.tolist()])
def cutmix(x, y):
    lam = np.random.beta(1.0, 1.0)
    w = _w_cls[y]                                                         # partner probability by its class
    idx = torch.multinomial(w / w.sum(), len(y), replacement=True)
    r = math.sqrt(1 - lam); ch = cw = int(32 * r); cy, cx = np.random.randint(32), np.random.randint(32)
    y1, y2, x1, x2 = max(cy - ch // 2, 0), min(cy + ch // 2, 32), max(cx - cw // 2, 0), min(cx + cw // 2, 32)
    x = x.clone(); x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    return x, y, y[idx], 1 - (y2 - y1) * (x2 - x1) / 1024

per-class val accuracy: [0.98, 0.984, 0.988, 0.968, 0.996, 0.968, 0.996, 0.988, 0.996, 0.988] 
partner weights: [1.21, 1.05, 0.89, 1.69, 0.56, 1.69, 0.56, 0.89, 0.56, 0.89]


In [29]:
# ======================= Density-rescaled nearest-neighbour classifier =======================
# d_k(x) = mean cosine distance from x to its 5 nearest TRAINING images of class k.
# sigma_k = the same quantity for training images of class k among their own class (how spread out class k is).
# score_k = -d_k / sigma_k ** p   (p = 0: plain per-class 5-NN distance; larger p rescales sparse classes more). p chosen on VAL.
cls_idx = [torch.where(ytr_ncm == k)[0] for k in range(10)]
sigma = []
for k in range(10):
    Fk = Fn_tr[cls_idx[k]]
    s_ = Fk @ Fk.T; s_.fill_diagonal_(-2)                                  # exclude self
    sigma.append((1 - s_.topk(5, dim=1).values).mean())
sigma = torch.stack(sigma)
print("class spread sigma:", dict(zip(["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"], [round(v, 4) for v in sigma.tolist()])))

def class_dist(split, bs=500):
    f = F.normalize(FE[split], dim=1); out = []
    for i in range(0, len(f), bs):
        sim = f[i:i + bs] @ Fn_tr.T
        out.append(torch.stack([(1 - sim[:, cls_idx[k]].topk(5, dim=1).values).mean(1) for k in range(10)], 1))
    return torch.cat(out)
CD = {s: class_dist(s) for s in ("val", "test", "kaggle")}
def density(split, p): return -CD[split] / (sigma[None] ** p)

PS = [0.0, 0.5, 1.0, 2.0]
for p in PS:
    print(f"  p={p:<4}: val {acc(density('val', p), yva):.4f} | internal test {acc(density('test', p), yte_int):.4f}")
cv_de, p_de = crossfit(density, PS)
print(f"\n0.915 model: val {acc(P0['val'], yva):.4f} | internal test {acc(P0['test'], yte_int):.4f}   | 0.917 soup: val {acc(uniform['val'], yva):.4f} | internal test {acc(uniform['test'], yte_int):.4f}")
print(f"DENSITY k-NN (p={p_de}, chosen on val): cross-fitted val {cv_de:.4f} | internal test {acc(density('test', p_de), yte_int):.4f}")
print("DECISION:", "beats the 0.915 model on val by > 0.2 pt" if cv_de > acc(P0['val'], yva) + 0.002 else "no gain: keep the soup submission")

class spread sigma: {'plane': 0.0407, 'car': 0.0272, 'bird': 0.0482, 'cat': 0.0559, 'deer': 0.0454, 'dog': 0.049, 'frog': 0.0378, 'horse': 0.0357, 'ship': 0.0304, 'truck': 0.0282}
  p=0.0 : val 0.9856 | internal test 0.9808
  p=0.5 : val 0.9828 | internal test 0.9796
  p=1.0 : val 0.9792 | internal test 0.9772
  p=2.0 : val 0.9624 | internal test 0.9652

0.915 model: val 0.9852 | internal test 0.9812   | 0.917 soup: val 0.9868 | internal test 0.9824
DENSITY k-NN (p=0.0, chosen on val): cross-fitted val 0.9856 | internal test 0.9808
DECISION: no gain: keep the soup submission


In [30]:
# ======================= Boost weak classes: log P_k + b * (1 - recall_k), applied to the 0.917 SOUP =======================
# recall_k = the soup's per-class accuracy on VAL; one strength b chosen on VAL (cross-fitted). b = 0 is the soup itself.
_ps = uniform["val"].argmax(1)
recall = torch.stack([(_ps[yva == k] == k).float().mean() for k in range(10)])
print("soup per-class val recall:", [round(v, 3) for v in recall.tolist()])
def boost(split, b):
    return uniform[split].clamp_min(1e-12).log() + b * (1 - recall)[None]
BS_ = [0.0, 0.5, 1.0, 2.0, 5.0, 10.0]
for b in BS_:
    flips = (boost("val", b).argmax(1) != boost("val", 0.0).argmax(1)).sum().item()
    print(f"  b={b:<5}: val {acc(boost('val', b), yva):.4f} | internal test {acc(boost('test', b), yte_int):.4f} | val predictions changed: {flips}")
cv_bo, b_bo = crossfit(boost, BS_)
print(f"\nSOUP: val {acc(uniform['val'], yva):.4f} | internal test {acc(uniform['test'], yte_int):.4f}")
print(f"BOOST (b={b_bo}, chosen on val): cross-fitted val {cv_bo:.4f} | internal test {acc(boost('test', b_bo), yte_int):.4f}")
if b_bo != 0.0 and cv_bo > acc(uniform["val"], yva) + 0.002:
    kp = boost("kaggle", b_bo).argmax(1).cpu().numpy(); kb = uniform["kaggle"].argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    drive_sub = os.path.join(RUN_DIR, f"submission_soup_boost_b{b_bo}_cvval{cv_bo:.4f}.csv"); sub.to_csv(drive_sub, index=False)
    print(f"DECISION: use it. saved {drive_sub} | Kaggle predictions changed vs the 0.917 soup: {(kp != kb).sum()} of {len(kp)}")
    if IN_COLAB:
        from google.colab import files; files.download(drive_sub)
else:
    print("DECISION: keep the 0.917 soup. No file written.")

soup per-class val recall: [0.984, 0.984, 0.992, 0.968, 1.0, 0.972, 0.996, 0.988, 0.996, 0.988]
  b=0.0  : val 0.9868 | internal test 0.9824 | val predictions changed: 0
  b=0.5  : val 0.9868 | internal test 0.9824 | val predictions changed: 0
  b=1.0  : val 0.9868 | internal test 0.9820 | val predictions changed: 0
  b=2.0  : val 0.9868 | internal test 0.9820 | val predictions changed: 0
  b=5.0  : val 0.9868 | internal test 0.9812 | val predictions changed: 0
  b=10.0 : val 0.9864 | internal test 0.9796 | val predictions changed: 1

SOUP: val 0.9868 | internal test 0.9824
BOOST (b=0.0, chosen on val): cross-fitted val 0.9868 | internal test 0.9824
DECISION: keep the 0.917 soup. No file written.


In [34]:
# ======================= Per-class temperature balancing: logits[c] * (S_good_mean / S_c) ** g =======================
# S_c = mean logit of class c on VAL images correctly predicted as c (0.915 model, flip-averaged features).
Zva = net.fc(FE["val"]); pred_va = Zva.argmax(1)
S = torch.stack([Zva[(pred_va == k) & (yva == k), k].mean() for k in range(10)])
_rec = torch.stack([(pred_va[yva == k] == k).float().mean() for k in range(10)])
S_good = S[_rec.argsort()[-5:]].mean()
tau = S_good / S
print("mean correct-class logit S_c:", [round(v, 2) for v in S.tolist()], "\ntau_c:", [round(v, 3) for v in tau.tolist()])

def tbal(split, p):
    shift, g = p
    z = net.fc(FE[split])
    if shift: z = z - z.min(1, keepdim=True).values                     # >= 0 so the multiplier acts the same on every class
    return z * (tau ** g)[None]
GT = [(s_, g) for s_ in (False, True) for g in (0.0, 0.5, 1.0, 2.0)]
for p in GT:
    flips = (tbal("val", p).argmax(1) != Zva.argmax(1)).sum().item()
    print(f"  shift={p[0]!s:5s} g={p[1]:<4}: val {acc(tbal('val', p), yva):.4f} | internal test {acc(tbal('test', p), yte_int):.4f} | val predictions changed: {flips}")
cv_tb, p_tb = crossfit(tbal, GT)
print(f"\n0.915 model: val {acc(P0['val'], yva):.4f} | internal test {acc(P0['test'], yte_int):.4f}   | 0.917 soup: val {acc(uniform['val'], yva):.4f} | internal test {acc(uniform['test'], yte_int):.4f}")
print(f"TEMPERATURE BALANCING ({p_tb}, chosen on val): cross-fitted val {cv_tb:.4f} | internal test {acc(tbal('test', p_tb), yte_int):.4f}")
print("DECISION:", "beats the 0.915 model on val by > 0.2 pt" if cv_tb > acc(P0['val'], yva) + 0.002 else "no gain: keep the soup submission")

mean correct-class logit S_c: [7.47, 6.69, 7.6, 7.34, 7.28, 7.12, 7.4, 7.16, 7.0, 7.1] 
tau_c: [0.962, 1.074, 0.945, 0.979, 0.987, 1.009, 0.972, 1.004, 1.027, 1.012]
  shift=False g=0.0 : val 0.9852 | internal test 0.9812 | val predictions changed: 0
  shift=False g=0.5 : val 0.9856 | internal test 0.9808 | val predictions changed: 1
  shift=False g=1.0 : val 0.9860 | internal test 0.9800 | val predictions changed: 2
  shift=False g=2.0 : val 0.9860 | internal test 0.9792 | val predictions changed: 2
  shift=True  g=0.0 : val 0.9852 | internal test 0.9812 | val predictions changed: 0
  shift=True  g=0.5 : val 0.9856 | internal test 0.9804 | val predictions changed: 1
  shift=True  g=1.0 : val 0.9860 | internal test 0.9796 | val predictions changed: 2
  shift=True  g=2.0 : val 0.9860 | internal test 0.9800 | val predictions changed: 5

0.915 model: val 0.9852 | internal test 0.9812   | 0.917 soup: val 0.9868 | internal test 0.9824
TEMPERATURE BALANCING ((False, 1.0), chosen on val): cro

In [35]:
# ======================= SVD low-rank tightening of the 0.917 SOUP's conv layers (no training) =======================
# Each conv weight (out, in/groups, k, k) -> (out, in*k*k) matrix; zero the smallest q fraction of singular values; rebuild.
# The final linear layer and BatchNorm are left untouched. q chosen on VAL (cross-fitted).
import copy
SOUP3 = {k: (sum(SD[n][k].float() for n in SD) / len(SD)) if next(iter(SD.values()))[k].is_floating_point() else next(iter(SD.values()))[k]
         for k in next(iter(SD.values()))}
conv_keys = [k for k, v in SOUP3.items() if v.ndim == 4]
print(f"{len(conv_keys)} conv layers")

def tightened(q):
    sd = dict(SOUP3)
    if q > 0:
        for k in conv_keys:
            Wt = SOUP3[k]; M = Wt.reshape(Wt.shape[0], -1)
            U_, S_, Vh_ = torch.linalg.svd(M, full_matrices=False)
            n_cut = int(round(q * len(S_)))
            if n_cut > 0: S_ = S_.clone(); S_[-n_cut:] = 0
            sd[k] = (U_ @ torch.diag(S_) @ Vh_).reshape(Wt.shape)
    m = copy.deepcopy(net); m.load_state_dict(sd); m.eval(); return m

QS = [0.0, 0.02, 0.05, 0.10]
SVDP = {q: {s: flip_probs(tightened(q), X) for s, X in {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}.items()} for q in QS}
for q in QS:
    print(f"  cut {q:.0%}: val {acc(SVDP[q]['val'], yva):.4f} | internal test {acc(SVDP[q]['test'], yte_int):.4f}")
cv_sv, q_sv = crossfit(lambda split, q: SVDP[q][split], QS)
print(f"\n0.917 SOUP (cut 0%): val {acc(SVDP[0.0]['val'], yva):.4f} | internal test {acc(SVDP[0.0]['test'], yte_int):.4f}   (expected 0.9868 / 0.9824)")
print(f"SVD TIGHTENING (cut {q_sv:.0%}, chosen on val): cross-fitted val {cv_sv:.4f} | internal test {acc(SVDP[q_sv]['test'], yte_int):.4f}")
if q_sv > 0 and cv_sv > acc(SVDP[0.0]["val"], yva) + 0.002:
    kp = SVDP[q_sv]["kaggle"].argmax(1).cpu().numpy(); kb = SVDP[0.0]["kaggle"].argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    drive_sub = os.path.join(RUN_DIR, f"submission_soup_svd{q_sv}_cvval{cv_sv:.4f}.csv"); sub.to_csv(drive_sub, index=False)
    print(f"DECISION: use it. saved {drive_sub} | predictions changed vs the 0.917 soup: {(kp != kb).sum()} of {len(kp)}")
else:
    print("DECISION: keep the 0.917 soup. No file written.")

31 conv layers
  cut 0%: val 0.9868 | internal test 0.9824
  cut 2%: val 0.9864 | internal test 0.9824
  cut 5%: val 0.9856 | internal test 0.9820
  cut 10%: val 0.9856 | internal test 0.9828

0.917 SOUP (cut 0%): val 0.9868 | internal test 0.9824   (expected 0.9868 / 0.9824)
SVD TIGHTENING (cut 0%, chosen on val): cross-fitted val 0.9868 | internal test 0.9824
DECISION: keep the 0.917 soup. No file written.


In [37]:
# ======================= Per-class range normalisation: (logit_i - min_i) / (max_i - min_i), stats from VAL =======================
# min_i / max_i = min and max of logit i over VAL images correctly predicted as class i. Blend strength g chosen on VAL (cross-fitted);
# note the stats themselves come from val, so the val score is slightly optimistic -> internal test is the honest check.
Zv = net.fc(FE["val"]); pv_ = Zv.argmax(1)
mn = torch.stack([Zv[(pv_ == k) & (yva == k), k].min() for k in range(10)])
mx = torch.stack([Zv[(pv_ == k) & (yva == k), k].max() for k in range(10)])
rng = (mx - mn)
print("per-class correct-logit range:", [round(v, 2) for v in rng.tolist()])
def rnorm(split, g):
    z = net.fc(FE[split])
    zn = (z - mn[None]) / rng[None] * rng.mean()                         # rescaled to the average range
    return (1 - g) * z + g * zn
GRN = [0.0, 0.25, 0.5, 1.0]
for g in GRN:
    flips = (rnorm("val", g).argmax(1) != Zv.argmax(1)).sum().item()
    print(f"  g={g:<4}: val {acc(rnorm('val', g), yva):.4f} | internal test {acc(rnorm('test', g), yte_int):.4f} | val predictions changed: {flips}")
cv_rn, g_rn = crossfit(rnorm, GRN)
print(f"\n0.915 model: val {acc(P0['val'], yva):.4f} | internal test {acc(P0['test'], yte_int):.4f}   | 0.917 soup: val {acc(uniform['val'], yva):.4f} | internal test {acc(uniform['test'], yte_int):.4f}")
print(f"RANGE NORMALISATION (g={g_rn}, chosen on val): cross-fitted val {cv_rn:.4f} | internal test {acc(rnorm('test', g_rn), yte_int):.4f}")
print("DECISION:", "beats the 0.915 model on val by > 0.2 pt" if cv_rn > acc(P0['val'], yva) + 0.002 else "no gain: keep the soup submission")

per-class correct-logit range: [8.81, 4.39, 7.72, 7.41, 8.45, 8.18, 6.69, 6.57, 7.66, 6.37]
  g=0.0 : val 0.9852 | internal test 0.9812 | val predictions changed: 0
  g=0.25: val 0.9856 | internal test 0.9812 | val predictions changed: 3
  g=0.5 : val 0.9852 | internal test 0.9808 | val predictions changed: 4
  g=1.0 : val 0.9848 | internal test 0.9796 | val predictions changed: 9

0.915 model: val 0.9852 | internal test 0.9812   | 0.917 soup: val 0.9868 | internal test 0.9824
RANGE NORMALISATION (g=0.25, chosen on val): cross-fitted val 0.9852 | internal test 0.9812
DECISION: no gain: keep the soup submission


In [38]:
# ---- Soup + BatchNorm recompute with TRAINING-MATCHED inputs (the BN step of SWA, Sparse Model Soups, REPAIR) ----
# Paste in the same notebook, after the soup cell (uses its variables). No training, no test images: train split only.
# Why it differs from the earlier Precise BN: in training, BN saw batches of [CutMixed, augmented image x 4 rotations]
# (SLA). The earlier Precise BN fed upright images only. Here the inputs follow the training pipeline.
import copy, os, re, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T

for _n in ["ROOT", "BASE", "RUN_DIR", "WORK_DIR", "CFG", "best", "X_full", "y_full", "sp", "to_dev", "normalize", "MEAN", "STD",
           "build_model", "count_params", "flip_probs", "Xva", "yva", "Xte_int", "yte_int", "Xkaggle", "test_ids", "SAMPLE_SUB",
           "device", "AMP_DTYPE"]:
    assert _n in globals(), f"missing {_n}: run the soup cell (setup part) first in this notebook"

# ---- training split (the 45k the model was trained on) ----
tr_idx = sp["train"] if "train" in sp.files else np.setdiff1d(np.arange(len(y_full)), np.concatenate([sp["val"], sp["test"]]))
Xtr = to_dev(X_full[tr_idx])
assert len(set(tr_idx.tolist()) & set(np.concatenate([sp["val"], sp["test"]]).tolist())) == 0
B, N_ROT = CFG["BATCH_SIZE"], CFG.get("SLA_ROTATIONS", 4)
MIX_PROB = CFG.get("MIX_PROB", CFG.get("CUTMIX_P", 0.0))
print(f"train images {len(Xtr)} | batch {B} | rotations {N_ROT} | CutMix prob {MIX_PROB} | TrivialAug {CFG.get('TRIVIAL_AUG')} "
      f"| grayscale p {CFG.get('GRAYSCALE_P', 0.0)} | cutout {CFG.get('CUTOUT')} | mixup share {CFG.get('MIXUP_SHARE', 0.0)}")

# ---- the 0.915 training augmentation (copied from the training notebook) ----
_trivial = T.TrivialAugmentWide()
def photometric(x_u8):
    if CFG.get("TRIVIAL_AUG", False):
        g = CFG.get("TA_GROUP", 8); x_u8 = torch.cat([_trivial(c) for c in x_u8.split(g)])
    p = CFG.get("GRAYSCALE_P", 0.0)
    if p > 0:
        gray = (x_u8.float() * torch.tensor([0.299, 0.587, 0.114], device=x_u8.device).view(1, 3, 1, 1)).sum(1, keepdim=True)
        gray = gray.round().clamp(0, 255).to(torch.uint8).expand(-1, 3, -1, -1)
        m = (torch.rand(len(x_u8), device=x_u8.device) < p)[:, None, None, None]; x_u8 = torch.where(m, gray, x_u8)
    return x_u8

def train_augment(x_u8):
    x_u8 = photometric(x_u8)
    x = (x_u8.float() / 255 - MEAN) / STD
    n, dev = x.size(0), x.device
    xp = F.pad(x, (4, 4, 4, 4), mode="reflect")
    dy = torch.randint(0, 9, (n,), device=dev); dx = torch.randint(0, 9, (n,), device=dev); ar = torch.arange(32, device=dev)
    x = xp[torch.arange(n, device=dev)[:, None, None, None], torch.arange(3, device=dev)[None, :, None, None],
           (dy[:, None] + ar)[:, None, :, None], (dx[:, None] + ar)[:, None, None, :]]
    flip = torch.rand(n, device=dev) < 0.5; x = torch.where(flip[:, None, None, None], x.flip(3), x)
    if CFG.get("CUTOUT", 0) > 0:
        cy = torch.randint(0, 32, (n,), device=dev); cx = torch.randint(0, 32, (n,), device=dev); h = CFG["CUTOUT"] // 2
        m = ((ar[None, :, None] - cy[:, None, None]).abs() < h) & ((ar[None, None, :] - cx[:, None, None]).abs() < h)
        x = x * (~m)[:, None].float()
    return x

def cutmix_x(x):
    lam = np.random.beta(1.0, 1.0); idx = torch.randperm(x.size(0), device=x.device)
    H, W = x.size(2), x.size(3); r = np.sqrt(1.0 - lam); ch, cw = int(H * r), int(W * r)
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2, x1, x2 = max(cy - ch // 2, 0), min(cy + ch // 2, H), max(cx - cw // 2, 0), min(cx + cw // 2, W)
    x = x.clone(); x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]; return x

@torch.no_grad()
def recompute_bn(model, aug, cutmix, rot, seed=0):
    """Same weights; BN running mean/var replaced by an exact average over one pass of the 45k train images."""
    m = copy.deepcopy(model)
    bns = [x for x in m.modules() if isinstance(x, nn.modules.batchnorm._BatchNorm)]
    for b in bns: b.reset_running_stats(); b.momentum = None            # None -> exact cumulative average
    m.train(); torch.manual_seed(seed); np.random.seed(seed)
    perm = torch.randperm(len(Xtr), device=device)
    for i in range(0, len(Xtr) - B + 1, B):                               # full batches only, as in training
        xu = Xtr[perm[i:i + B]]
        x = train_augment(xu) if aug else (xu.float() / 255 - MEAN) / STD
        if cutmix and MIX_PROB > 0 and np.random.rand() < MIX_PROB: x = cutmix_x(x)
        if rot: x = torch.cat([torch.rot90(x, k, dims=(2, 3)) for k in range(N_ROT)])
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            m(x.contiguous(memory_format=torch.channels_last))
    m.eval(); return m

acc = lambda s, y: (s.argmax(1) == y).float().mean().item()
def score(m, kaggle=False):
    r = {"val": acc(flip_probs(m, Xva), yva), "test": acc(flip_probs(m, Xte_int), yte_int)}
    if kaggle: r["kaggle_pred"] = flip_probs(m, Xkaggle).argmax(1).cpu().numpy()
    return r

# ---- models: the 0.915 single model and the 0.917 uniform soup of 3 ----
single = build_model(CFG).to(device).to(memory_format=torch.channels_last); single.load_state_dict(best["model"]); single.eval()
SOUP3 = [BASE, BASE + "_ft100_lr0.02", BASE + "_tail50_noheavyaug"]
sds = [torch.load(os.path.join(ROOT, "runs", r, "best.pt"), map_location=device, weights_only=False)["model"] for r in SOUP3]
soup = copy.deepcopy(single)
soup.load_state_dict({k: (sum(s[k].float() for s in sds) / len(sds)).to(sds[0][k].dtype) if sds[0][k].is_floating_point() else sds[0][k]
                      for k in sds[0]}); soup.eval()
print(f"params {count_params(soup):,} (BN recompute changes buffers only, not parameters)")

VARIANTS = {
    "A. aug + CutMix + 4 rotations (training-matched)": dict(aug=True, cutmix=True, rot=True),
    "B. aug + 4 rotations (no CutMix)":                 dict(aug=True, cutmix=False, rot=True),
    "C. clean + 4 rotations":                           dict(aug=False, cutmix=False, rot=True),
}
t0 = time.time(); rows = []
s0 = score(single); rows.append(("0.915 single, saved BN", s0["val"], s0["test"]))
sA = score(recompute_bn(single, **VARIANTS["A. aug + CutMix + 4 rotations (training-matched)"]))
rows.append(("0.915 single, recompute A (sanity check)", sA["val"], sA["test"]))
base = score(soup, kaggle=True); rows.append(("SOUP, averaged BN (the 0.917 file)", base["val"], base["test"]))
res = {}
for name, kw in VARIANTS.items():
    for seed in ((0, 1) if name.startswith("A") else (0,)):
        r = score(recompute_bn(soup, seed=seed, **kw), kaggle=True); res[(name, seed)] = r
        rows.append((f"SOUP, recompute {name} seed {seed}", r["val"], r["test"]))
df = pd.DataFrame(rows, columns=["model", "val", "internal_test"])
print(df.assign(val=df.val.round(4), internal_test=df.internal_test.round(4)).to_string(index=False))
print(f"(done in {time.time() - t0:.0f}s)  expected soup averaged BN: 0.9868 / 0.9824 | single: 0.9852 / 0.9812")

# ---- decision on VAL only: must beat the soup (averaged BN) by > 0.2 pt (5 of 2,500 images) ----
(kbest, rbest) = max(res.items(), key=lambda kv: kv[1]["val"])
print(f"\nbest on VAL: {kbest[0]} seed {kbest[1]}: val {rbest['val']:.4f} | internal test {rbest['test']:.4f} "
      f"| change vs soup: val {100 * (rbest['val'] - base['val']):+.2f} pt, internal {int(round((rbest['test'] - base['test']) * 2500)):+d} images")
if rbest["val"] > base["val"] + 0.002:
    name, seed = kbest
    m = recompute_bn(soup, seed=seed, **VARIANTS[name])
    kp = flip_probs(m, Xkaggle).argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    tag = f"soup3_bn{name[0]}_s{seed}_val{rbest['val']:.4f}_test{rbest['test']:.4f}"
    torch.save({"model": m.state_dict(), "cfg": CFG, "norm": best["norm"], "soup_of": SOUP3, "bn": name},
               os.path.join(RUN_DIR, f"best_{tag}.pt"))
    sub.to_csv(os.path.join(RUN_DIR, f"submission_{tag}.csv"), index=False)
    os.makedirs(os.path.join(WORK_DIR, "soupbn"), exist_ok=True); local = os.path.join(WORK_DIR, "soupbn", "submission.csv"); sub.to_csv(local, index=False)
    print(f"DECISION: use it. saved {os.path.join(RUN_DIR, f'submission_{tag}.csv')}\n"
          f"Kaggle predictions that differ from the 0.917 soup: {(kp != base['kaggle_pred']).sum()} of {len(kp)}")
    if globals().get("IN_COLAB"):
        from google.colab import files; files.download(local)
else:
    print("DECISION: keep the 0.917 soup (no recompute beats it on VAL by > 0.2 pt). No file written.")

train images 45000 | batch 256 | rotations 4 | CutMix prob 0.5 | TrivialAug True | grayscale p 0.1 | cutout 16 | mixup share 0.0
params 4,998,738 (BN recompute changes buffers only, not parameters)
                                                                  model    val  internal_test
                                                 0.915 single, saved BN 0.9852         0.9812
                               0.915 single, recompute A (sanity check) 0.9848         0.9812
                                     SOUP, averaged BN (the 0.917 file) 0.9868         0.9824
SOUP, recompute A. aug + CutMix + 4 rotations (training-matched) seed 0 0.9864         0.9820
SOUP, recompute A. aug + CutMix + 4 rotations (training-matched) seed 1 0.9868         0.9820
                SOUP, recompute B. aug + 4 rotations (no CutMix) seed 0 0.9868         0.9824
                          SOUP, recompute C. clean + 4 rotations seed 0 0.9756         0.9760
(done in 100s)  expected soup averaged BN: 0.9868 

In [39]:
# ---- OPTION 3: fine-tune the 0.915 model with a 4-model TEACHER ENSEMBLE (SE-WRN + 0.915 + ft100 + tail50) ----
# Paste in the same notebook AFTER the soup cell and the BN-recompute cell (uses their variables: model code, data,
# Xtr, train_augment, flip_probs, ...). Same recipe as the 0.915 run: SLA+SD, KD alpha 0.9, ASAM 0.5, EMA 0.999,
# TrivialAug + crop/flip + cutout + CutMix 50%, label smoothing 0.1, SGD-Nesterov, cosine. Only the teacher changes.
# GATE: stops before training unless the teacher ensemble beats the soup on VAL by > 0.2 pt.
import os, math, time, copy, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

for _n in ["Xtr", "tr_idx", "y_full", "train_augment", "flip_probs", "build_model", "forward_both", "normalize", "CFG", "best", "ROOT", "BASE",
           "Xva", "yva", "Xte_int", "yte_int", "device", "AMP_DTYPE", "MEAN", "STD"]:
    assert _n in globals(), f"missing {_n}: run the soup cell and the BN-recompute cell first in this notebook"

ytr = torch.from_numpy(y_full[tr_idx]).to(device)                        # labels of the 45k train images (tr_idx from the BN cell)
RUN_NAME = BASE + "_ing3kd"; OUT = os.path.join(ROOT, "runs", RUN_NAME); os.makedirs(OUT, exist_ok=True)
EPOCHS, LR, SEED = 20, 0.01, 31
GATE_VAL = 0.9888                                   # soup val 0.9868 + 0.2 pt
TEACHERS = ["teacher_sewrn28_10_sla250_v1", BASE, BASE + "_ft100_lr0.02", BASE + "_tail50_noheavyaug"]
BS, N_ROT = CFG["BATCH_SIZE"], CFG.get("SLA_ROTATIONS", 4)
KD_ALPHA, KD_T, SD_W = CFG["KD_ALPHA"], CFG["KD_T"], CFG.get("SLA_SD_WEIGHT", 1.0)
RHO, ASAM_ETA = CFG["SAM_RHO"], 0.01
MIX_P, CM_ALPHA = CFG.get("MIX_PROB", CFG.get("CUTMIX_P", 0.0)), CFG.get("CUTMIX_ALPHA", 1.0)
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"recipe: KD alpha {KD_ALPHA}, T {KD_T} | SD weight {SD_W} | ASAM rho {RHO} (adaptive {CFG.get('SAM_ADAPTIVE')}) | EMA {CFG['EMA_DECAY']} "
      f"| CutMix {MIX_P} | LS {CFG['LABEL_SMOOTHING']} | wd {CFG['WEIGHT_DECAY']} | batch {BS} | {EPOCHS} epochs, LR {LR}")

# ---- 1. teachers: load, calibrate each on VAL (temperature), equal-weight average, one ensemble temperature ----
@torch.no_grad()
def logits_of(m, X, flip=False, bs=500):
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs]); xb = xb.flip(3) if flip else xb
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None): out.append(m(xb).float())
    return torch.cat(out)
def fit_T(lp, y, lo=0.1, hi=10.0):
    grid = torch.exp(torch.linspace(math.log(lo), math.log(hi), 300, device=lp.device))
    return float(grid[torch.stack([F.cross_entropy(lp / t, y) for t in grid]).argmin()])

teachers, Ts, pv, pt = [], [], [], []
for rn in TEACHERS:
    ck = torch.load(os.path.join(ROOT, "runs", rn, "best.pt"), map_location=device, weights_only=False)
    t = build_model(ck["cfg"]).to(device).to(memory_format=torch.channels_last); t.load_state_dict(ck["model"]); t.eval()
    for p in t.parameters(): p.requires_grad_(False)
    T = fit_T(logits_of(t, Xva), yva)
    v = 0.5 * ((logits_of(t, Xva) / T).softmax(1) + (logits_of(t, Xva, True) / T).softmax(1))
    s = 0.5 * ((logits_of(t, Xte_int) / T).softmax(1) + (logits_of(t, Xte_int, True) / T).softmax(1))
    teachers.append(t); Ts.append(T); pv.append(v); pt.append(s)
    print(f"  teacher {rn[-45:]:45s} T {T:.3f} | val {(v.argmax(1) == yva).float().mean():.4f} | internal test {(s.argmax(1) == yte_int).float().mean():.4f}")
ens_v, ens_t = sum(pv) / len(pv), sum(pt) / len(pt)
ENS_T = fit_T(torch.log(ens_v.clamp_min(1e-12)), yva, 0.3, 3.0)
ev, et = (ens_v.argmax(1) == yva).float().mean().item(), (ens_t.argmax(1) == yte_int).float().mean().item()
print(f"TEACHER ENSEMBLE (equal weights, nothing chosen on val): val {ev:.4f} | internal test {et:.4f} | ensemble T {ENS_T:.3f}")
if ev <= GATE_VAL:
    raise RuntimeError(f"GATE: ensemble val {ev:.4f} is not above {GATE_VAL} (soup val + 0.2 pt). A student copying it is "
                       "unlikely to beat the soup. Stopping before training (nothing lost).")
print("GATE passed -> training")

@torch.no_grad()
def teacher_probs(xb):
    p = 0
    with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
        for t, T in zip(teachers, Ts): p = p + F.softmax(t(xb).float() / T, 1) / len(teachers)
    return F.softmax(torch.log(p.clamp_min(1e-12)) / (ENS_T * KD_T), 1)

# ---- 2. student = the 0.915 weights, same optimiser / schedule / EMA as the soup ingredients ----
model = build_model(CFG).to(device).to(memory_format=torch.channels_last); model.load_state_dict(best["model"])
assert abs((flip_probs(model.eval(), Xva).argmax(1) == yva).float().mean().item() - 0.9852) < 0.0015, "start point is not the 0.915 model"
decay = [p for p in model.parameters() if p.ndim > 1]; no_decay = [p for p in model.parameters() if p.ndim <= 1]
opt = torch.optim.SGD([{"params": decay, "weight_decay": CFG["WEIGHT_DECAY"]}, {"params": no_decay, "weight_decay": 0.0}],
                      lr=LR, momentum=CFG.get("MOMENTUM", 0.9), nesterov=True)
steps = EPOCHS * math.ceil(len(Xtr) / BS); warm = math.ceil(len(Xtr) / BS)
sched = torch.optim.lr_scheduler.SequentialLR(opt, [torch.optim.lr_scheduler.LinearLR(opt, 0.01, 1.0, warm),
                                                    torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps - warm, 0.0)], [warm])
ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(CFG["EMA_DECAY"]), use_buffers=True)
ce = nn.CrossEntropyLoss(label_smoothing=CFG["LABEL_SMOOTHING"])
params = [p for p in model.parameters() if p.requires_grad]

def cutmix(x, y):
    lam = np.random.beta(CM_ALPHA, CM_ALPHA); idx = torch.randperm(x.size(0), device=x.device)
    H, W = x.size(2), x.size(3); r = math.sqrt(1.0 - lam); ch, cw = int(H * r), int(W * r)
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2, x1, x2 = max(cy - ch // 2, 0), min(cy + ch // 2, H), max(cx - cw // 2, 0), min(cx + cw // 2, W)
    x = x.clone(); x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    return x, y, y[idx], 1.0 - (y2 - y1) * (x2 - x1) / (H * W)

def sla_loss(xb, yb, mixed, ya, yb2, lam, tp):
    B = len(yb); rot = torch.arange(N_ROT, device=device).repeat_interleave(B)
    xr = torch.cat([torch.rot90(xb, k, dims=(2, 3)) for k in range(N_ROT)]).contiguous(memory_format=torch.channels_last)
    with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
        sd_lo, j_lo = forward_both(model, xr)
    sd_lo, j_lo = sd_lo.float(), j_lo.float()
    lj = (lam * ce(j_lo, ya.repeat(N_ROT) * N_ROT + rot) + (1 - lam) * ce(j_lo, yb2.repeat(N_ROT) * N_ROT + rot)) if mixed \
        else ce(j_lo, yb.repeat(N_ROT) * N_ROT + rot)
    jv = j_lo.view(N_ROT, B, 10, N_ROT); agg = torch.stack([jv[k, :, :, k] for k in range(N_ROT)]).mean(0)
    sd0 = sd_lo[:B]
    sd = F.kl_div(F.log_softmax(sd0, 1), F.softmax(agg.detach(), 1), reduction="batchmean")
    kd = F.kl_div(F.log_softmax(sd0 / KD_T, 1), tp, reduction="batchmean") * KD_T ** 2
    return sd0, lj + SD_W * (KD_ALPHA * kd + (1 - KD_ALPHA) * sd)

# ---- 3. train ----
best_val, hist = -1.0, []
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); perm = torch.randperm(len(Xtr), device=device); tot = 0.0
    for i in range(0, len(Xtr), BS):
        idx = perm[i:i + BS]; yb = ytr[idx]
        xb = train_augment(Xtr[idx]).contiguous(memory_format=torch.channels_last)
        mixed = MIX_P > 0 and np.random.rand() < MIX_P; ya = yb2 = None; lam = 1.0
        if mixed: xb, ya, yb2, lam = cutmix(xb, yb)
        tp = teacher_probs(xb.contiguous(memory_format=torch.channels_last))
        opt.zero_grad(set_to_none=True); _, loss = sla_loss(xb, yb, mixed, ya, yb2, lam, tp); loss.backward()
        with torch.no_grad():                                   # ASAM (Kwon et al. 2021), as in the 0.915 run
            if CFG.get("SAM_ADAPTIVE", True):
                tw = [(p.abs() + ASAM_ETA) if p.grad is not None else None for p in params]
                gn = torch.norm(torch.stack([(t * p.grad).norm() for t, p in zip(tw, params) if t is not None])) + 1e-12
                eps = [t * t * p.grad * (RHO / gn) if t is not None else None for t, p in zip(tw, params)]
            else:
                gn = torch.norm(torch.stack([p.grad.norm() for p in params if p.grad is not None])) + 1e-12
                eps = [p.grad * (RHO / gn) if p.grad is not None else None for p in params]
            for p, e in zip(params, eps):
                if e is not None: p.add_(e)
        opt.zero_grad(set_to_none=True); _, loss2 = sla_loss(xb, yb, mixed, ya, yb2, lam, tp); loss2.backward()
        with torch.no_grad():
            for p, e in zip(params, eps):
                if e is not None: p.sub_(e)
        opt.step(); sched.step(); ema.update_parameters(model); tot += loss.item() * len(idx)
    net = ema.module.eval()
    va = (flip_probs(net, Xva).argmax(1) == yva).float().mean().item()
    te = (flip_probs(net, Xte_int).argmax(1) == yte_int).float().mean().item() if (ep + 1) % 5 == 0 or ep + 1 == EPOCHS else None
    if va > best_val:
        best_val = va
        torch.save({"model": net.state_dict(), "cfg": CFG, "epoch": ep + 1, "val_acc": va, "norm": best["norm"], "is_ema": True,
                    "teachers": TEACHERS}, os.path.join(OUT, "best.pt"))
    if (ep + 1) % 5 == 0: torch.save({"model": net.state_dict(), "epoch": ep + 1}, os.path.join(OUT, f"snap_e{ep + 1:03d}.pt"))
    sec = time.time() - t0; hist.append({"epoch": ep + 1, "loss": tot / len(Xtr), "val": va, "internal_test": te, "sec": sec})
    pd.DataFrame(hist).to_csv(os.path.join(OUT, "history.csv"), index=False)
    print(f"ep {ep + 1:2d}/{EPOCHS} | loss {tot / len(Xtr):.4f} | val ema {va:.4f}{' *' if va == best_val else '  '} | "
          f"{sec:.0f}s | eta {(EPOCHS - ep - 1) * sec / 60:.0f} min" + (f" | INTERNAL TEST {te:.4f} (0.915: 0.9812, soup: 0.9824)" if te is not None else ""), flush=True)
print(f"DONE: best val {best_val:.4f} -> {os.path.join(OUT, 'best.pt')}. Now rerun the soup cell (it picks up _ing3kd automatically).")

recipe: KD alpha 0.9, T 1.0 | SD weight 1.0 | ASAM rho 0.5 (adaptive True) | EMA 0.999 | CutMix 0.5 | LS 0.1 | wd 0.0005 | batch 256 | 20 epochs, LR 0.01
  teacher teacher_sewrn28_10_sla250_v1                  T 0.948 | val 0.9880 | internal test 0.9840
  teacher d_sewrn_only_student_resnext29_slasd_400ep_v1 T 0.812 | val 0.9852 | internal test 0.9812
  teacher student_resnext29_slasd_400ep_v1_ft100_lr0.02 T 0.838 | val 0.9876 | internal test 0.9828
  teacher nt_resnext29_slasd_400ep_v1_tail50_noheavyaug T 0.825 | val 0.9852 | internal test 0.9800
TEACHER ENSEMBLE (equal weights, nothing chosen on val): val 0.9864 | internal test 0.9848 | ensemble T 0.882


RuntimeError: GATE: ensemble val 0.9864 is not above 0.9888 (soup val + 0.2 pt). A student copying it is unlikely to beat the soup. Stopping before training (nothing lost).

In [40]:
# ---- Teacher-guided last-layer fine-tune of the 0.917 SOUP (one network; only the final 10-class layer changes) ----
# Paste after the soup, BN and option-3 cells (uses soup, teachers[0] = SE-WRN, Ts, train_augment, Xtr, ytr, flip_probs).
# Backbone frozen (eval mode, BN untouched). The fc layer learns to match the SE-WRN teacher on augmented TRAIN images.
# Decision on VAL only (> soup + 0.2 pt); internal test is the honest check. ~1 min per epoch on an L4 (estimate).
import os, math, time, copy, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
for _n in ["soup", "teachers", "Ts", "train_augment", "Xtr", "ytr", "flip_probs", "Xkaggle", "test_ids", "SAMPLE_SUB", "RUN_DIR", "WORK_DIR"]:
    assert _n in globals(), f"missing {_n}: run the soup, BN and option-3 cells first"
teacher, T_t = teachers[0], Ts[0]                       # SE-WRN-28-10 + SLA, calibrated temperature from option-3 cell
HEAD_EPOCHS, HEAD_LR, ALPHA, BSZ = 4, 0.01, 0.9, 512
torch.manual_seed(0); np.random.seed(0)
acc = lambda p, y: (p.argmax(1) == y).float().mean().item()

stu = copy.deepcopy(soup).eval()
for p in stu.parameters(): p.requires_grad_(False)
for p in stu.fc.parameters(): p.requires_grad_(True)
fc0 = copy.deepcopy(stu.fc.state_dict())
opt = torch.optim.SGD(stu.fc.parameters(), lr=HEAD_LR, momentum=0.9, nesterov=True, weight_decay=5e-4)
steps = HEAD_EPOCHS * math.ceil(len(Xtr) / BSZ); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
ce = nn.CrossEntropyLoss(label_smoothing=0.1)

base_v, base_t = acc(flip_probs(stu, Xva), yva), acc(flip_probs(stu, Xte_int), yte_int)
rows = [("soup (start)", 0, base_v, base_t)]; states = {0: copy.deepcopy(fc0)}
for ep in range(1, HEAD_EPOCHS + 1):
    t0 = time.time(); perm = torch.randperm(len(Xtr), device=device)
    for i in range(0, len(Xtr), BSZ):
        idx = perm[i:i + BSZ]; xb = train_augment(Xtr[idx]).contiguous(memory_format=torch.channels_last); yb = ytr[idx]
        with torch.no_grad(), torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            f = stu.drop(stu.features(xb)).float()
            tp = F.softmax(teacher(xb).float() / T_t, 1)
        lo = stu.fc(f)
        loss = ALPHA * F.kl_div(F.log_softmax(lo, 1), tp, reduction="batchmean") + (1 - ALPHA) * ce(lo, yb)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
    v, t = acc(flip_probs(stu, Xva), yva), acc(flip_probs(stu, Xte_int), yte_int)
    rows.append((f"head fine-tune epoch {ep}", ep, v, t)); states[ep] = copy.deepcopy(stu.fc.state_dict())
    print(f"epoch {ep}: val {v:.4f} | internal test {t:.4f} | {time.time() - t0:.0f}s", flush=True)
df = pd.DataFrame(rows, columns=["model", "epoch", "val", "internal_test"]); print(df.round(4).to_string(index=False))

# decision on VAL only: best epoch must beat the soup by > 0.2 pt (5 of 2,500)
b = df.iloc[1:].sort_values(["val", "epoch"], ascending=[False, True]).iloc[0]
print(f"best on VAL: epoch {int(b.epoch)}: val {b.val:.4f} (soup {base_v:.4f}) | internal {b.internal_test:.4f} (soup {base_t:.4f}, "
      f"{int(round((b.internal_test - base_t) * 2500)):+d} images)")
if b.val > base_v + 0.002:
    stu.fc.load_state_dict(states[int(b.epoch)]); stu.eval()
    kp = flip_probs(stu, Xkaggle).argmax(1).cpu().numpy(); k0 = flip_probs(soup, Xkaggle).argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    tag = f"soup3_headkd_e{int(b.epoch)}_val{b.val:.4f}_test{b.internal_test:.4f}"
    torch.save({"model": stu.state_dict(), "cfg": CFG, "norm": best["norm"]}, os.path.join(RUN_DIR, f"best_{tag}.pt"))
    sub.to_csv(os.path.join(RUN_DIR, f"submission_{tag}.csv"), index=False)
    os.makedirs(os.path.join(WORK_DIR, "headkd"), exist_ok=True); local = os.path.join(WORK_DIR, "headkd", "submission.csv"); sub.to_csv(local, index=False)
    print(f"DECISION: use it -> {os.path.join(RUN_DIR, f'submission_{tag}.csv')} | Kaggle predictions changed vs soup: {(kp != k0).sum()} of {len(kp)}")
    if globals().get("IN_COLAB"):
        from google.colab import files; files.download(local)
else:
    print("DECISION: keep the 0.917 soup (no epoch beats it on VAL by > 0.2 pt). No file written.")

epoch 1: val 0.9864 | internal test 0.9820 | 13s
epoch 2: val 0.9868 | internal test 0.9820 | 12s
epoch 3: val 0.9864 | internal test 0.9824 | 12s
epoch 4: val 0.9864 | internal test 0.9824 | 12s
                 model  epoch    val  internal_test
          soup (start)      0 0.9868         0.9824
head fine-tune epoch 1      1 0.9864         0.9820
head fine-tune epoch 2      2 0.9868         0.9820
head fine-tune epoch 3      3 0.9864         0.9824
head fine-tune epoch 4      4 0.9864         0.9824
best on VAL: epoch 2: val 0.9868 (soup 0.9868) | internal 0.9820 (soup 0.9824, -1 images)
DECISION: keep the 0.917 soup (no epoch beats it on VAL by > 0.2 pt). No file written.


In [41]:
# ---- Zoom TTA on the 0.917 SOUP: add scaled-in / scaled-out views (one network, no training) ----
# Paste after the soup and BN cells (uses soup, normalize, Xva, yva, Xte_int, yte_int, Xkaggle, ...).
# Zoom in  (s > 1): upscale bilinear, centre-crop 32x32.  Zoom out (s < 1): downscale, reflect-pad back to 32x32.
# Every view also flipped. Setting CHOSEN ON VAL ONLY (must beat flip-only soup by > 0.2 pt); internal test = honest check.
import os, time, numpy as np, pandas as pd, torch, torch.nn.functional as F
for _n in ["soup", "normalize", "Xva", "yva", "Xte_int", "yte_int", "Xkaggle", "test_ids", "SAMPLE_SUB", "RUN_DIR", "WORK_DIR", "device", "AMP_DTYPE"]:
    assert _n in globals(), f"missing {_n}: run the soup and BN cells first"

def zoom(x, s):
    if s == 1.0: return x
    n = int(round(32 * s))
    y = F.interpolate(x, size=(n, n), mode="bilinear", align_corners=False, antialias=s < 1)
    if s > 1:
        o = (n - 32) // 2; return y[:, :, o:o + 32, o:o + 32]
    p = 32 - n; return F.pad(y, (p // 2, p - p // 2, p // 2, p - p // 2), mode="reflect")

@torch.no_grad()
def view_probs(m, X, s, bs=500):
    out = []
    for i in range(0, len(X), bs):
        xb = zoom(normalize(X[i:i + bs]), s).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            out.append((m(xb).float().softmax(1) + m(xb.flip(3)).float().softmax(1)) / 2)
    return torch.cat(out)

SCALES = [0.75, 0.875, 1.0, 1.125, 1.25]
SETTINGS = {
    "flip only (the 0.917 soup)":      [1.0],
    "+ zoom in 1.125":                 [1.0, 1.125],
    "+ zoom out 0.875":                [1.0, 0.875],
    "+ zoom in/out 1.125, 0.875":      [1.0, 1.125, 0.875],
    "+ zoom in 1.125, 1.25":           [1.0, 1.125, 1.25],
    "+ zoom out 0.875, 0.75":          [1.0, 0.875, 0.75],
    "+ all (0.75 ... 1.25)":           [0.75, 0.875, 1.0, 1.125, 1.25],
}
soup.eval(); t0 = time.time()
DATA = {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}
C = {k: {s: view_probs(soup, X, s) for s in SCALES} for k, X in DATA.items()}
acc = lambda p, y: (p.argmax(1) == y).float().mean().item()
P = lambda k, st: sum(C[k][s] for s in SETTINGS[st])
for s in SCALES: print(f"  single view scale {s:5.3f}: val {acc(C['val'][s], yva):.4f} | internal test {acc(C['test'][s], yte_int):.4f}")
rows = [(st, len(v), acc(P("val", st), yva), acc(P("test", st), yte_int)) for st, v in SETTINGS.items()]
df = pd.DataFrame(rows, columns=["setting", "scales", "val", "internal_test"])
print(df.round(4).to_string(index=False)); print(f"({time.time() - t0:.0f}s)")

base = df.iloc[0]; cand = df.iloc[1:].sort_values(["val", "scales"], ascending=[False, True]).iloc[0]
print(f"\nbest on VAL: {cand.setting}: val {cand.val:.4f} (soup {base.val:.4f}) | internal {cand.internal_test:.4f} "
      f"(soup {base.internal_test:.4f}, {int(round((cand.internal_test - base.internal_test) * 2500)):+d} images)")
if cand.val > base.val + 0.002:
    kp = P("kaggle", cand.setting).argmax(1).cpu().numpy(); k0 = P("kaggle", base.setting).argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    tag = "soup3_zoomtta_" + "_".join(str(s) for s in SETTINGS[cand.setting]) + f"_val{cand.val:.4f}_test{cand.internal_test:.4f}"
    sub.to_csv(os.path.join(RUN_DIR, f"submission_{tag}.csv"), index=False)
    os.makedirs(os.path.join(WORK_DIR, "zoomtta"), exist_ok=True); local = os.path.join(WORK_DIR, "zoomtta", "submission.csv"); sub.to_csv(local, index=False)
    print(f"DECISION: use it -> {os.path.join(RUN_DIR, f'submission_{tag}.csv')} | Kaggle predictions changed vs soup: {(kp != k0).sum()} of {len(kp)}")
    if globals().get("IN_COLAB"):
        from google.colab import files; files.download(local)
else:
    print("DECISION: keep the 0.917 soup (no zoom setting beats it on VAL by > 0.2 pt). No file written.")

  single view scale 0.750: val 0.9664 | internal test 0.9600
  single view scale 0.875: val 0.9768 | internal test 0.9760
  single view scale 1.000: val 0.9868 | internal test 0.9824
  single view scale 1.125: val 0.9732 | internal test 0.9716
  single view scale 1.250: val 0.9600 | internal test 0.9572
                   setting  scales    val  internal_test
flip only (the 0.917 soup)       1 0.9868         0.9824
           + zoom in 1.125       2 0.9836         0.9820
          + zoom out 0.875       2 0.9856         0.9812
+ zoom in/out 1.125, 0.875       3 0.9828         0.9820
     + zoom in 1.125, 1.25       3 0.9780         0.9776
    + zoom out 0.875, 0.75       3 0.9824         0.9788
     + all (0.75 ... 1.25)       5 0.9812         0.9796
(4s)

best on VAL: + zoom out 0.875: val 0.9856 (soup 0.9868) | internal 0.9812 (soup 0.9824, -3 images)
DECISION: keep the 0.917 soup (no zoom setting beats it on VAL by > 0.2 pt). No file written.


In [42]:
# ---- Zoom TTA on the 0.915 SINGLE model (reuses zoom(), view_probs(), SETTINGS from the zoom cell) ----
for _n in ["single", "view_probs", "SETTINGS", "SCALES"]:
    assert _n in globals(), f"missing {_n}: run the BN cell and the zoom cell first"
single.eval(); t0 = time.time()
C1 = {k: {s: view_probs(single, X, s) for s in SCALES} for k, X in DATA.items()}
P1 = lambda k, st: sum(C1[k][s] for s in SETTINGS[st])
for s in SCALES: print(f"  single view scale {s:5.3f}: val {acc(C1['val'][s], yva):.4f} | internal test {acc(C1['test'][s], yte_int):.4f}")
rows = [(st.replace("the 0.917 soup", "the 0.915 model"), len(v), acc(P1("val", st), yva), acc(P1("test", st), yte_int)) for st, v in SETTINGS.items()]
df1 = pd.DataFrame(rows, columns=["setting", "scales", "val", "internal_test"])
print(df1.round(4).to_string(index=False)); print(f"({time.time() - t0:.0f}s)  expected flip only: 0.9852 / 0.9812")

base1 = df1.iloc[0]; cand1 = df1.iloc[1:].sort_values(["val", "scales"], ascending=[False, True]).iloc[0]
st1 = list(SETTINGS)[df1.index.get_loc(cand1.name)]
print(f"\nbest on VAL: {cand1.setting}: val {cand1.val:.4f} (0.915 {base1.val:.4f}) | internal {cand1.internal_test:.4f} "
      f"(0.915 {base1.internal_test:.4f}, {int(round((cand1.internal_test - base1.internal_test) * 2500)):+d} images)")
if cand1.val > base1.val + 0.002:
    kp = P1("kaggle", st1).argmax(1).cpu().numpy(); k0 = P1("kaggle", list(SETTINGS)[0]).argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    tag = "single0915_zoomtta_" + "_".join(str(s) for s in SETTINGS[st1]) + f"_val{cand1.val:.4f}_test{cand1.internal_test:.4f}"
    sub.to_csv(os.path.join(RUN_DIR, f"submission_{tag}.csv"), index=False)
    os.makedirs(os.path.join(WORK_DIR, "zoomtta"), exist_ok=True); local = os.path.join(WORK_DIR, "zoomtta", "submission_0915.csv"); sub.to_csv(local, index=False)
    print(f"DECISION: use it -> {os.path.join(RUN_DIR, f'submission_{tag}.csv')} | Kaggle predictions changed vs 0.915: {(kp != k0).sum()} of {len(kp)}")
    if globals().get("IN_COLAB"):
        from google.colab import files; files.download(local)
else:
    print("DECISION: keep the 0.915 flip-only predictions (no zoom setting beats them on VAL by > 0.2 pt). No file written.")

  single view scale 0.750: val 0.9660 | internal test 0.9580
  single view scale 0.875: val 0.9756 | internal test 0.9740
  single view scale 1.000: val 0.9852 | internal test 0.9812
  single view scale 1.125: val 0.9740 | internal test 0.9708
  single view scale 1.250: val 0.9568 | internal test 0.9520
                    setting  scales    val  internal_test
flip only (the 0.915 model)       1 0.9852         0.9812
            + zoom in 1.125       2 0.9840         0.9824
           + zoom out 0.875       2 0.9844         0.9820
 + zoom in/out 1.125, 0.875       3 0.9824         0.9808
      + zoom in 1.125, 1.25       3 0.9780         0.9764
     + zoom out 0.875, 0.75       3 0.9816         0.9800
      + all (0.75 ... 1.25)       5 0.9808         0.9788
(4s)  expected flip only: 0.9852 / 0.9812

best on VAL: + zoom out 0.875: val 0.9844 (0.915 0.9852) | internal 0.9820 (0.915 0.9812, +2 images)
DECISION: keep the 0.915 flip-only predictions (no zoom setting beats them on VAL by > 

In [43]:
# ---- Blur / sharpen TTA on the 0.917 SOUP and the 0.915 SINGLE model (one network each, no training) ----
# Paste after the soup, BN and zoom cells (uses soup, single, normalize, DATA, acc, ...).
# blur: Gaussian (sigma), sharpen: unsharp mask x + a*(x - blur(x)). Every view also flipped.
# Setting CHOSEN ON VAL ONLY (must beat flip-only by > 0.2 pt); internal test = honest check.
import os, re, time, numpy as np, pandas as pd, torch, torch.nn.functional as F
for _n in ["soup", "single", "normalize", "Xva", "yva", "Xte_int", "yte_int", "Xkaggle", "test_ids", "SAMPLE_SUB", "RUN_DIR", "WORK_DIR"]:
    assert _n in globals(), f"missing {_n}: run the soup and BN cells first"

def gblur(x, sigma):
    r = 2; k = torch.exp(-torch.arange(-r, r + 1, device=x.device, dtype=x.dtype) ** 2 / (2 * sigma ** 2)); k = k / k.sum()
    c = x.size(1); x = F.pad(x, (r, r, r, r), mode="reflect")
    x = F.conv2d(x, k.view(1, 1, 1, -1).repeat(c, 1, 1, 1), groups=c)
    return F.conv2d(x, k.view(1, 1, -1, 1).repeat(c, 1, 1, 1), groups=c)

OPS = {"none": lambda x: x,
       "blur s0.5": lambda x: gblur(x, 0.5), "blur s0.8": lambda x: gblur(x, 0.8),
       "sharpen a0.5": lambda x: x + 0.5 * (x - gblur(x, 1.0)), "sharpen a1.0": lambda x: x + 1.0 * (x - gblur(x, 1.0))}
SET_B = {"flip only": ["none"],
         "+ blur 0.5": ["none", "blur s0.5"], "+ blur 0.8": ["none", "blur s0.8"],
         "+ sharpen 0.5": ["none", "sharpen a0.5"], "+ sharpen 1.0": ["none", "sharpen a1.0"],
         "+ blur 0.5 + sharpen 0.5": ["none", "blur s0.5", "sharpen a0.5"],
         "+ blur 0.8 + sharpen 1.0": ["none", "blur s0.8", "sharpen a1.0"]}

@torch.no_grad()
def op_probs(m, X, op, bs=500):
    out = []
    for i in range(0, len(X), bs):
        xb = OPS[op](normalize(X[i:i + bs]).float()).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            out.append((m(xb).float().softmax(1) + m(xb.flip(3)).float().softmax(1)) / 2)
    return torch.cat(out)

acc = lambda p, y: (p.argmax(1) == y).float().mean().item()
DATA = {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}
for mname, m in [("0.917 SOUP", soup), ("0.915 SINGLE", single)]:
    m.eval(); t0 = time.time()
    CB = {k: {op: op_probs(m, X, op) for op in OPS} for k, X in DATA.items()}
    PB = lambda k, st: sum(CB[k][op] for op in SET_B[st])
    print(f"\n===== {mname} =====")
    for op in OPS: print(f"  single view {op:13s}: val {acc(CB['val'][op], yva):.4f} | internal test {acc(CB['test'][op], yte_int):.4f}")
    rows = [(st, len(v), acc(PB("val", st), yva), acc(PB("test", st), yte_int)) for st, v in SET_B.items()]
    dfb = pd.DataFrame(rows, columns=["setting", "views", "val", "internal_test"]); print(dfb.round(4).to_string(index=False))
    b0 = dfb.iloc[0]; cb = dfb.iloc[1:].sort_values(["val", "views"], ascending=[False, True]).iloc[0]
    print(f"best on VAL: {cb.setting}: val {cb.val:.4f} (flip only {b0.val:.4f}) | internal {cb.internal_test:.4f} "
          f"(flip only {b0.internal_test:.4f}, {int(round((cb.internal_test - b0.internal_test) * 2500)):+d} images) | {time.time() - t0:.0f}s")
    if cb.val > b0.val + 0.002:
        kp = PB("kaggle", cb.setting).argmax(1).cpu().numpy(); k0 = PB("kaggle", "flip only").argmax(1).cpu().numpy()
        sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
        assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
        tag = re.sub(r"[^a-z0-9.]+", "_", f"{mname}_blurtta_{cb.setting}".lower()).strip("_") + f"_val{cb.val:.4f}_test{cb.internal_test:.4f}"
        sub.to_csv(os.path.join(RUN_DIR, f"submission_{tag}.csv"), index=False)
        print(f"DECISION: use it -> {os.path.join(RUN_DIR, f'submission_{tag}.csv')} | Kaggle predictions changed: {(kp != k0).sum()} of {len(kp)}")
    else:
        print("DECISION: keep flip only (no blur/sharpen setting beats it on VAL by > 0.2 pt). No file written.")



===== 0.917 SOUP =====
  single view none         : val 0.9868 | internal test 0.9824
  single view blur s0.5    : val 0.9832 | internal test 0.9792
  single view blur s0.8    : val 0.9488 | internal test 0.9376
  single view sharpen a0.5 : val 0.9860 | internal test 0.9796
  single view sharpen a1.0 : val 0.9812 | internal test 0.9792
                 setting  views    val  internal_test
               flip only      1 0.9868         0.9824
              + blur 0.5      2 0.9856         0.9820
              + blur 0.8      2 0.9780         0.9756
           + sharpen 0.5      2 0.9852         0.9816
           + sharpen 1.0      2 0.9852         0.9816
+ blur 0.5 + sharpen 0.5      3 0.9868         0.9824
+ blur 0.8 + sharpen 1.0      3 0.9812         0.9804
best on VAL: + blur 0.5 + sharpen 0.5: val 0.9868 (flip only 0.9868) | internal 0.9824 (flip only 0.9824, +0 images) | 4s
DECISION: keep flip only (no blur/sharpen setting beats it on VAL by > 0.2 pt). No file written.

===== 0.9

In [44]:
# ---- Bilateral-filter TTA (edge-preserving smoothing) on the 0.917 SOUP and the 0.915 SINGLE model (no training) ----
# Paste after the soup and BN cells (uses soup, single, MEAN, STD, Xva, yva, Xte_int, yte_int, Xkaggle, ...).
# Filter on pixels in [0,1], 5x5 window, spatial sigma 1.0; range sigma = how different a neighbour's colour may be and
# still be averaged (small = only near-identical pixels are smoothed, edges kept). Every view also flipped.
# Setting CHOSEN ON VAL ONLY (must beat flip-only by > 0.2 pt); internal test = honest check.
import os, re, time, numpy as np, pandas as pd, torch, torch.nn.functional as F
for _n in ["soup", "single", "MEAN", "STD", "Xva", "yva", "Xte_int", "yte_int", "Xkaggle", "test_ids", "SAMPLE_SUB", "RUN_DIR"]:
    assert _n in globals(), f"missing {_n}: run the soup and BN cells first"

def bilateral(x, sigma_s=1.0, sigma_r=0.1, r=2):
    """x: N,3,H,W in [0,1]. Joint (all 3 channels) range weight, Gaussian spatial weight."""
    N, C, H, W = x.shape; k = 2 * r + 1
    patches = F.unfold(F.pad(x, (r, r, r, r), mode="reflect"), k).view(N, C, k * k, H * W)
    centre = x.view(N, C, 1, H * W)
    ax = torch.arange(-r, r + 1, device=x.device, dtype=x.dtype)
    ws = torch.exp(-(ax[:, None] ** 2 + ax[None, :] ** 2) / (2 * sigma_s ** 2)).view(1, 1, k * k, 1)
    wr = torch.exp(-((patches - centre) ** 2).sum(1, keepdim=True) / (2 * sigma_r ** 2))
    w = ws * wr
    return ((patches * w).sum(2) / w.sum(2)).view(N, C, H, W)

OPS = {"none": None, "bilat r0.05": 0.05, "bilat r0.10": 0.10, "bilat r0.20": 0.20}
SET_L = {"flip only": ["none"],
         "+ bilat 0.05": ["none", "bilat r0.05"], "+ bilat 0.10": ["none", "bilat r0.10"], "+ bilat 0.20": ["none", "bilat r0.20"],
         "+ bilat 0.05 + 0.10": ["none", "bilat r0.05", "bilat r0.10"]}

@torch.no_grad()
def bil_probs(m, X, op, bs=500):
    out = []
    for i in range(0, len(X), bs):
        x = X[i:i + bs].float() / 255
        if OPS[op] is not None: x = bilateral(x, 1.0, OPS[op])
        xb = ((x - MEAN) / STD).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            out.append((m(xb).float().softmax(1) + m(xb.flip(3)).float().softmax(1)) / 2)
    return torch.cat(out)

acc = lambda p, y: (p.argmax(1) == y).float().mean().item()
DATA = {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}
for mname, m in [("0.917 SOUP", soup), ("0.915 SINGLE", single)]:
    m.eval(); t0 = time.time()
    CL = {k: {op: bil_probs(m, X, op) for op in OPS} for k, X in DATA.items()}
    PL = lambda k, st: sum(CL[k][op] for op in SET_L[st])
    print(f"\n===== {mname} =====")
    for op in OPS: print(f"  single view {op:12s}: val {acc(CL['val'][op], yva):.4f} | internal test {acc(CL['test'][op], yte_int):.4f}")
    rows = [(st, len(v), acc(PL("val", st), yva), acc(PL("test", st), yte_int)) for st, v in SET_L.items()]
    dfl = pd.DataFrame(rows, columns=["setting", "views", "val", "internal_test"]); print(dfl.round(4).to_string(index=False))
    b0 = dfl.iloc[0]; cl = dfl.iloc[1:].sort_values(["val", "views"], ascending=[False, True]).iloc[0]
    print(f"best on VAL: {cl.setting}: val {cl.val:.4f} (flip only {b0.val:.4f}) | internal {cl.internal_test:.4f} "
          f"(flip only {b0.internal_test:.4f}, {int(round((cl.internal_test - b0.internal_test) * 2500)):+d} images) | {time.time() - t0:.0f}s")
    if cl.val > b0.val + 0.002:
        kp = PL("kaggle", cl.setting).argmax(1).cpu().numpy(); k0 = PL("kaggle", "flip only").argmax(1).cpu().numpy()
        sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
        assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
        tag = re.sub(r"[^a-z0-9.]+", "_", f"{mname}_bilateral_{cl.setting}".lower()).strip("_") + f"_val{cl.val:.4f}_test{cl.internal_test:.4f}"
        sub.to_csv(os.path.join(RUN_DIR, f"submission_{tag}.csv"), index=False)
        print(f"DECISION: use it -> {os.path.join(RUN_DIR, f'submission_{tag}.csv')} | Kaggle predictions changed: {(kp != k0).sum()} of {len(kp)}")
    else:
        print("DECISION: keep flip only (no bilateral setting beats it on VAL by > 0.2 pt). No file written.")


===== 0.917 SOUP =====
  single view none        : val 0.9868 | internal test 0.9824
  single view bilat r0.05 : val 0.9800 | internal test 0.9792
  single view bilat r0.10 : val 0.9752 | internal test 0.9712
  single view bilat r0.20 : val 0.9576 | internal test 0.9624
            setting  views    val  internal_test
          flip only      1 0.9868         0.9824
       + bilat 0.05      2 0.9856         0.9824
       + bilat 0.10      2 0.9836         0.9804
       + bilat 0.20      2 0.9800         0.9784
+ bilat 0.05 + 0.10      3 0.9832         0.9820
best on VAL: + bilat 0.05: val 0.9856 (flip only 0.9868) | internal 0.9824 (flip only 0.9824, +0 images) | 4s
DECISION: keep flip only (no bilateral setting beats it on VAL by > 0.2 pt). No file written.

===== 0.915 SINGLE =====
  single view none        : val 0.9852 | internal test 0.9812
  single view bilat r0.05 : val 0.9800 | internal test 0.9812
  single view bilat r0.10 : val 0.9724 | internal test 0.9684
  single view bila

In [2]:
# ============================================================================================================
# STANDALONE: colour + size TTA on the 0.917 SOUP and the 0.915 SINGLE model (no training). One cell, fresh runtime OK.
# Works on Colab (mounts Drive MyDrive/cifar10_resnet_challenge), Kaggle, HPC. Chosen on VAL only (> +0.2 pt).
# ============================================================================================================
import os, re, glob, time, json, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
try:
    import google.colab  # noqa
    IN_COLAB = True
    from google.colab import drive; drive.mount("/content/drive")
    ROOT, WORK_DIR = "/content/drive/MyDrive/cifar10_resnet_challenge", "/content/work"
except ImportError:
    IN_COLAB = False
    if os.path.exists("/kaggle/working/cifar10_resnet_challenge"): ROOT, WORK_DIR = "/kaggle/working/cifar10_resnet_challenge", "/kaggle/working/work"
    else: ROOT = os.environ.get("SOUP_ROOT", "/scratch/ds8223/cifar10_resnet_challenge"); WORK_DIR = os.environ.get("SOUP_WORK", "/scratch/ds8223/kd_work")
BASE = "kd_sewrn_only_student_resnext29_slasd_400ep_v1"
RUN_DIR = os.path.join(ROOT, "runs", BASE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if device.type == "cpu" or torch.cuda.get_device_capability()[0] >= 8 else None
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (slow)")

# ---- Models (same code as the training notebooks): ResNet, ResNeXt-29, Shake-Shake, chosen with CFG["ARCH"] ----
class BasicBlock(nn.Module):
    """Post-activation basic block (our original ResNet)."""
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Identity()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x), inplace=True)

class CifarResNet(nn.Module):
    def __init__(self, widths=(64, 128, 256), blocks=(3, 3, 3), num_classes=10, dropout=0.0):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        layers, cin = [], widths[0]
        for i, (w, nb) in enumerate(zip(widths, blocks)):
            for j in range(nb):
                layers.append(BasicBlock(cin, w, 2 if (j == 0 and i > 0) else 1)); cin = w
        self.layers = nn.Sequential(*layers)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        for b in self.layers: nn.init.zeros_(b.bn2.weight)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)

class ResNeXtBlock(nn.Module):
    """ResNeXt bottleneck: 1x1 conv -> grouped 3x3 conv (cardinality groups) -> 1x1 conv, plus skip (Xie et al. 2017)."""
    def __init__(self, cin, inner, cout, stride, cardinality):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, inner, 1, bias=False); self.bn1 = nn.BatchNorm2d(inner)
        self.conv2 = nn.Conv2d(inner, inner, 3, stride, 1, groups=cardinality, bias=False); self.bn2 = nn.BatchNorm2d(inner)
        self.conv3 = nn.Conv2d(inner, cout, 1, bias=False); self.bn3 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Identity()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = F.relu(self.bn2(self.conv2(out)), inplace=True)
        out = self.bn3(self.conv3(out))
        return F.relu(out + self.shortcut(x), inplace=True)

class ResNeXt29(nn.Module):
    """CIFAR ResNeXt-29: 3x3 stem (64) + 3 stages x 3 blocks; output widths 256/512/1024;
    inner width = cardinality * bottleneck_width, doubled at each stage. (2, 64) is the standard 2x64d."""
    def __init__(self, cardinality=2, bottleneck_width=64, blocks=(3, 3, 3), out_widths=(256, 512, 1024),
                 num_classes=10, dropout=0.0, stem=64, strides=(1, 2, 2)):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, stem, 3, 1, 1, bias=False), nn.BatchNorm2d(stem), nn.ReLU(inplace=True))
        layers, cin = [], stem
        for i, (cout, nb) in enumerate(zip(out_widths, blocks)):
            inner = cardinality * bottleneck_width * (2 ** i)
            for j in range(nb):
                layers.append(ResNeXtBlock(cin, inner, cout, strides[i] if j == 0 else 1, cardinality)); cin = cout
        self.layers = nn.Sequential(*layers)
        self.drop = nn.Dropout(dropout); self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        for b in self.layers: nn.init.zeros_(b.bn3.weight)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)

class SE(nn.Module):
    """Squeeze-excitation (Hu et al. 2018): channel weights from global average pooling. Teachers only."""
    def __init__(self, c, r=16):
        super().__init__()
        self.fc1, self.fc2 = nn.Conv2d(c, c // r, 1), nn.Conv2d(c // r, c, 1)
    def forward(self, x):
        return x * torch.sigmoid(self.fc2(F.relu(self.fc1(F.adaptive_avg_pool2d(x, 1)))))

class WBlock(nn.Module):
    """Pre-activation wide residual block (Zagoruyko & Komodakis 2016)."""
    def __init__(self, cin, cout, stride, drop, se_r=0):
        super().__init__()
        self.bn1, self.conv1 = nn.BatchNorm2d(cin), nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn2, self.conv2 = nn.BatchNorm2d(cout), nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.dropout = nn.Dropout(drop)
        self.skip = None if (stride == 1 and cin == cout) else nn.Conv2d(cin, cout, 1, stride, bias=False)
        if se_r: self.se = SE(cout, se_r)
    def forward(self, x):
        o = F.relu(self.bn1(x))
        sc = x if self.skip is None else self.skip(o)
        o = self.conv2(self.dropout(F.relu(self.bn2(self.conv1(o)))))
        if hasattr(self, "se"): o = self.se(o)
        return o + sc

class WRN(nn.Module):
    """WideResNet for CIFAR (Zagoruyko & Komodakis 2016). k = widen factor (WRN-28-10: 36.5M params) or explicit stage widths."""
    def __init__(self, depth=28, k=10, dropout=0.0, num_classes=10, se_r=0):
        super().__init__()
        n = (depth - 4) // 6
        w = [16] + (list(k) if isinstance(k, (tuple, list)) else [16 * k, 32 * k, 64 * k])   # k: widen factor, or the 3 stage widths
        self.stem = nn.Conv2d(3, w[0], 3, 1, 1, bias=False)
        layers, cin = [], w[0]
        for i in range(3):
            for j in range(n):
                layers.append(WBlock(cin, w[i + 1], 2 if (j == 0 and i > 0) else 1, dropout, se_r)); cin = w[i + 1]
        self.layers = nn.Sequential(*layers); self.bn = nn.BatchNorm2d(cin)
        self.drop = nn.Dropout(0.0); self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(F.relu(self.bn(self.layers(self.stem(x)))), 1), 1)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))

def _build_base(c):
    """Build any of our architectures from a config dict (also rebuilds teachers from their saved cfg)."""
    arch = c.get("ARCH", "resnet")
    if arch == "resnet":
        return CifarResNet(c["WIDTHS"], c["BLOCKS"], dropout=c.get("DROPOUT", 0.0))
    if arch == "resnext":
        return ResNeXt29(c.get("CARDINALITY", 2), c.get("BOTTLENECK_WIDTH", 64), c.get("BLOCKS", (3, 3, 3)),
                         c.get("WIDTHS", (256, 512, 1024)), dropout=c.get("DROPOUT", 0.0),
                         strides=tuple(c.get("STRIDES", (1, 2, 2))))   # (1, 1, 2) = higher-resolution student, same params
    if arch in ("wrn", "sewrn"):     # sewrn = WRN with SE blocks: a training-only teacher
        return WRN(c["DEPTH"], c["WIDEN"], c.get("DROPOUT", 0.0), se_r=c.get("SE_REDUCTION", 16) if arch == "sewrn" else 0)
    raise ValueError(f"unknown ARCH {arch}")

def build_model(c):
    """Base network; with SLA on, adds the joint (class x rotation) head used only during training."""
    net = _build_base(c)
    if c.get("SLA", False):
        net.fc_joint = nn.Linear(net.fc.in_features, 10 * c.get("SLA_ROTATIONS", 4))
    return net

def forward_both(net, x):
    """Returns (10-class self-distillation head logits, joint class x rotation head logits)."""
    f = net.drop(net.features(x))
    return net.fc(f), net.fc_joint(f)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

# ---- data, split, base model ----
best = torch.load(os.path.join(RUN_DIR, "best.pt"), map_location=device, weights_only=False); CFG = best["cfg"]
CACHE_DIR, SPLIT_DIR = os.path.join(ROOT, "data", "cache"), os.path.join(ROOT, "data", "splits")
X_full = np.load(os.path.join(CACHE_DIR, "train_x.npy")); y_full = np.load(os.path.join(CACHE_DIR, "train_y.npy"))
X_test = np.load(os.path.join(CACHE_DIR, "test_x.npy")); test_ids = np.load(os.path.join(CACHE_DIR, "test_ids.npy"))
SAMPLE_SUB = pd.read_csv(os.path.join(CACHE_DIR, "sample_submission.csv"))
sp = np.load(os.path.join(SPLIT_DIR, f"split_seed{CFG['SPLIT_SEED']}_val{CFG['VAL_FRAC']}_test{CFG['TEST_FRAC']}_n{len(y_full)}.npz"))
def to_dev(x): return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous().to(device)
Xva, yva = to_dev(X_full[sp["val"]]), torch.from_numpy(y_full[sp["val"]]).to(device)
Xte_int, yte_int = to_dev(X_full[sp["test"]]), torch.from_numpy(y_full[sp["test"]]).to(device)
Xkaggle = to_dev(X_test)
MEAN, STD = best["norm"]["mean"].to(device), best["norm"]["std"].to(device)
def normalize(x_u8): return ((x_u8.float() / 255 - MEAN) / STD).contiguous(memory_format=torch.channels_last)
net = build_model(CFG).to(device).to(memory_format=torch.channels_last); net.load_state_dict(best["model"]); net.eval()
acc = lambda s, y: (s.argmax(1) == y).float().mean().item()
DATA = {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}

@torch.no_grad()
def flip_probs(m, X, bs=500):                       # exactly how the 0.915 / 0.917 submissions were scored
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs])
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            out.append((m(xb).float().softmax(1) + m(xb.flip(3)).float().softmax(1)) / 2)
    return torch.cat(out)

# ---- the two models: 0.915 single and the 0.917 uniform soup of 3 ----
single = net
SOUP3 = [BASE, BASE + "_ft100_lr0.02", BASE + "_tail50_noheavyaug"]
sds = [torch.load(os.path.join(ROOT, "runs", r, "best.pt"), map_location=device, weights_only=False)["model"] for r in SOUP3]
soup = copy.deepcopy(single)
soup.load_state_dict({k: (sum(s[k].float() for s in sds) / len(sds)).to(sds[0][k].dtype) if sds[0][k].is_floating_point() else sds[0][k]
                      for k in sds[0]}); soup.eval(); del sds
print(f"0.915 single: val {acc(flip_probs(single, Xva), yva):.4f} | internal {acc(flip_probs(single, Xte_int), yte_int):.4f}  (expected 0.9852 / 0.9812)")
print(f"0.917 soup:   val {acc(flip_probs(soup, Xva), yva):.4f} | internal {acc(flip_probs(soup, Xte_int), yte_int):.4f}  (expected 0.9868 / 0.9824)")
print(f"params {count_params(soup):,}")

def zoom(x, s):
    if s == 1.0: return x
    n = int(round(32 * s))
    y = F.interpolate(x, size=(n, n), mode="bilinear", align_corners=False, antialias=s < 1)
    if s > 1:
        o = (n - 32) // 2; return y[:, :, o:o + 32, o:o + 32]
    p = 32 - n; return F.pad(y, (p // 2, p - p // 2, p // 2, p - p // 2), mode="reflect")

import itertools
import torchvision.transforms.functional as TF

COLOR = {"orig": lambda u: u,
         "gray": lambda u: TF.rgb_to_grayscale(u, 3),
         "sat0.5": lambda u: TF.adjust_saturation(u, 0.5), "sat1.5": lambda u: TF.adjust_saturation(u, 1.5),
         "hue-0.05": lambda u: TF.adjust_hue(u, -0.05), "hue+0.05": lambda u: TF.adjust_hue(u, 0.05)}
SIZES = [1.0, 0.875, 1.125]

@torch.no_grad()
def cs_probs(m, X, col, s, bs=500):
    out = []
    for i in range(0, len(X), bs):
        u = COLOR[col](X[i:i + bs])                                   # uint8, colour change in pixel space
        xb = zoom((u.float() / 255 - MEAN) / STD, s).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            out.append((m(xb).float().softmax(1) + m(xb.flip(3)).float().softmax(1)) / 2)
    return torch.cat(out)

SET_C = {"flip only": [("orig", 1.0)],
         "+ gray": [("orig", 1.0), ("gray", 1.0)],
         "+ saturation 0.5/1.5": [("orig", 1.0), ("sat0.5", 1.0), ("sat1.5", 1.0)],
         "+ hue -/+0.05": [("orig", 1.0), ("hue-0.05", 1.0), ("hue+0.05", 1.0)],
         "+ all colours": [(c, 1.0) for c in COLOR],
         "+ all colours + sizes 0.875/1.125": [(c, 1.0) for c in COLOR] + [("orig", 0.875), ("orig", 1.125)],
         "all colours x all sizes": list(itertools.product(COLOR, SIZES))}
need = sorted({v for vs in SET_C.values() for v in vs})
acc = lambda p, y: (p.argmax(1) == y).float().mean().item()
DATA = {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}
for mname, m in [("0.917 SOUP", soup), ("0.915 SINGLE", single)]:
    m.eval(); t0 = time.time()
    CC = {k: {v: cs_probs(m, X, *v) for v in need} for k, X in DATA.items()}
    PC = lambda k, st: sum(CC[k][v] for v in SET_C[st])
    print(f"\n===== {mname} =====")
    for v in need:
        if v[1] == 1.0: print(f"  single view {v[0]:9s}: val {acc(CC['val'][v], yva):.4f} | internal test {acc(CC['test'][v], yte_int):.4f}")
    rows = [(st, len(v), acc(PC("val", st), yva), acc(PC("test", st), yte_int)) for st, v in SET_C.items()]
    dfc = pd.DataFrame(rows, columns=["setting", "views", "val", "internal_test"]); print(dfc.round(4).to_string(index=False))
    b0 = dfc.iloc[0]; cc = dfc.iloc[1:].sort_values(["val", "views"], ascending=[False, True]).iloc[0]
    print(f"best on VAL: {cc.setting}: val {cc.val:.4f} (flip only {b0.val:.4f}) | internal {cc.internal_test:.4f} "
          f"(flip only {b0.internal_test:.4f}, {int(round((cc.internal_test - b0.internal_test) * 2500)):+d} images) | {time.time() - t0:.0f}s")
    if cc.val > b0.val + 0.002:
        kp = PC("kaggle", cc.setting).argmax(1).cpu().numpy(); k0 = PC("kaggle", "flip only").argmax(1).cpu().numpy()
        sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
        assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
        tag = re.sub(r"[^a-z0-9.]+", "_", f"{mname}_colsize_{cc.setting}".lower()).strip("_") + f"_val{cc.val:.4f}_test{cc.internal_test:.4f}"
        sub.to_csv(os.path.join(RUN_DIR, f"submission_{tag}.csv"), index=False)
        print(f"DECISION: use it -> {os.path.join(RUN_DIR, f'submission_{tag}.csv')} | Kaggle predictions changed: {(kp != k0).sum()} of {len(kp)}")
    else:
        print("DECISION: keep flip only (no colour/size setting beats it on VAL by > 0.2 pt). No file written.")

Mounted at /content/drive
device: NVIDIA A100-SXM4-40GB
0.915 single: val 0.9852 | internal 0.9812  (expected 0.9852 / 0.9812)
0.917 soup:   val 0.9868 | internal 0.9824  (expected 0.9868 / 0.9824)
params 4,998,738

===== 0.917 SOUP =====
  single view gray     : val 0.9612 | internal test 0.9560
  single view hue+0.05 : val 0.9796 | internal test 0.9780
  single view hue-0.05 : val 0.9840 | internal test 0.9784
  single view orig     : val 0.9868 | internal test 0.9824
  single view sat0.5   : val 0.9812 | internal test 0.9776
  single view sat1.5   : val 0.9844 | internal test 0.9816
                          setting  views    val  internal_test
                        flip only      1 0.9868         0.9824
                           + gray      2 0.9820         0.9780
             + saturation 0.5/1.5      3 0.9856         0.9796
                    + hue -/+0.05      3 0.9864         0.9804
                    + all colours      6 0.9860         0.9796
+ all colours + sizes 0.875/1

In [3]:
# ---- Original image only (1x, no flip) vs flip TTA (2x), on the 0.917 soup and the 0.915 single model ----
@torch.no_grad()
def plain_probs(m, X, bs=500):
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs])
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            out.append(m(xb).float().softmax(1))
    return torch.cat(out)

acc = lambda p, y: (p.argmax(1) == y).float().mean().item()
for mname, m in [("0.917 SOUP", soup), ("0.915 SINGLE", single)]:
    m.eval()
    pv, pt = plain_probs(m, Xva), plain_probs(m, Xte_int)
    fv, ft = flip_probs(m, Xva), flip_probs(m, Xte_int)
    print(f"{mname:13s} original (1x): val {acc(pv, yva):.4f} | internal test {acc(pt, yte_int):.4f}")
    print(f"{'':13s} flip TTA (2x): val {acc(fv, yva):.4f} | internal test {acc(ft, yte_int):.4f}")
    d_val, d_test = acc(pv, yva) - acc(fv, yva), acc(pt, yte_int) - acc(ft, yte_int)
    print(f"{'':13s} original minus flip: val {int(round(d_val * 2500)):+d} images | internal {int(round(d_test * 2500)):+d} images")
    if d_val > 0.002:
        kp = plain_probs(m, Xkaggle).argmax(1).cpu().numpy(); k0 = flip_probs(m, Xkaggle).argmax(1).cpu().numpy()
        sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
        assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all()
        tag = re.sub(r"[^a-z0-9.]+", "_", f"{mname}_noflip".lower()).strip("_") + f"_val{acc(pv, yva):.4f}_test{acc(pt, yte_int):.4f}"
        sub.to_csv(os.path.join(RUN_DIR, f"submission_{tag}.csv"), index=False)
        print(f"{'':13s} DECISION: original wins on VAL -> saved submission_{tag}.csv | Kaggle predictions changed: {(kp != k0).sum()}")
    else:
        print(f"{'':13s} DECISION: keep flip TTA (original does not beat it on VAL by > 0.2 pt)")

0.917 SOUP    original (1x): val 0.9840 | internal test 0.9808
              flip TTA (2x): val 0.9868 | internal test 0.9824
              original minus flip: val -7 images | internal -4 images
              DECISION: keep flip TTA (original does not beat it on VAL by > 0.2 pt)
0.915 SINGLE  original (1x): val 0.9856 | internal test 0.9816
              flip TTA (2x): val 0.9852 | internal test 0.9812
              original minus flip: val +1 images | internal +1 images
              DECISION: keep flip TTA (original does not beat it on VAL by > 0.2 pt)


In [4]:
# ---- Small-shift TTA (2 px left/right/up/down, reflect padding as in training), with and without flips ----
def shift(x, dy, dx, p=4):
    if dy == 0 and dx == 0: return x
    xp = F.pad(x, (p, p, p, p), mode="reflect")
    return xp[:, :, p - dy:p - dy + 32, p - dx:p - dx + 32].contiguous(memory_format=torch.channels_last)

POS5 = [(0, 0), (2, 0), (-2, 0), (0, 2), (0, -2)]

@torch.no_grad()
def view(m, X, dy, dx, flip, bs=500):
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs])
        if flip: xb = xb.flip(3)
        xb = shift(xb, dy, dx)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            out.append(m(xb).float().softmax(1))
    return torch.cat(out)

SETS = {"original (1x)":            [(0, 0, False)],
        "flip (2x, current)":       [(0, 0, False), (0, 0, True)],
        "shifts, no flip (5x)":     [(dy, dx, False) for dy, dx in POS5],
        "shifts + flips (10x)":     [(dy, dx, f) for dy, dx in POS5 for f in (False, True)]}
acc = lambda p, y: (p.argmax(1) == y).float().mean().item()
for mname, m in [("0.917 SOUP", soup), ("0.915 SINGLE", single)]:
    m.eval()
    need = sorted({v for vs in SETS.values() for v in vs})
    C = {k: {v: view(m, X, *v) for v in need} for k, X in {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}.items()}
    P = lambda k, st: sum(C[k][v] for v in SETS[st])
    print(f"\n===== {mname} =====")
    rows = [(st, acc(P("val", st), yva), acc(P("test", st), yte_int)) for st in SETS]
    for st, v, t in rows: print(f"  {st:22s} val {v:.4f} | internal test {t:.4f}")
    base_v, base_t = rows[1][1], rows[1][2]
    st, v, t = max([r for r in rows if r[0] != "flip (2x, current)"], key=lambda r: r[1])
    print(f"  best other on VAL: {st}: val {int(round((v - base_v) * 2500)):+d} images | internal {int(round((t - base_t) * 2500)):+d} images vs flip")
    if v > base_v + 0.002:
        kp = P("kaggle", st).argmax(1).cpu().numpy(); k0 = P("kaggle", "flip (2x, current)").argmax(1).cpu().numpy()
        sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
        assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all()
        tag = re.sub(r"[^a-z0-9.]+", "_", f"{mname}_{st}".lower()).strip("_") + f"_val{v:.4f}_test{t:.4f}"
        sub.to_csv(os.path.join(RUN_DIR, f"submission_{tag}.csv"), index=False)
        print(f"  DECISION: use it -> submission_{tag}.csv | Kaggle predictions changed: {(kp != k0).sum()}")
    else:
        print("  DECISION: keep flip (nothing beats it on VAL by > 0.2 pt)")


===== 0.917 SOUP =====
  original (1x)          val 0.9840 | internal test 0.9808
  flip (2x, current)     val 0.9868 | internal test 0.9824
  shifts, no flip (5x)   val 0.9832 | internal test 0.9832
  shifts + flips (10x)   val 0.9852 | internal test 0.9832
  best other on VAL: shifts + flips (10x): val -4 images | internal +2 images vs flip
  DECISION: keep flip (nothing beats it on VAL by > 0.2 pt)

===== 0.915 SINGLE =====
  original (1x)          val 0.9856 | internal test 0.9816
  flip (2x, current)     val 0.9852 | internal test 0.9812
  shifts, no flip (5x)   val 0.9832 | internal test 0.9824
  shifts + flips (10x)   val 0.9840 | internal test 0.9820
  best other on VAL: original (1x): val +1 images | internal +1 images vs flip
  DECISION: keep flip (nothing beats it on VAL by > 0.2 pt)


In [5]:
# ---- write the soup + shifts + flips (10x) submission ----
@torch.no_grad()
def tta10(m, X):
    return sum(view(m, X, dy, dx, f) for dy, dx in POS5 for f in (False, True))
soup.eval()
kp = tta10(soup, Xkaggle).argmax(1).cpu().numpy()
k0 = flip_probs(soup, Xkaggle).argmax(1).cpu().numpy()
sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
path = os.path.join(RUN_DIR, "submission_soup3_shift2_flip_10x_val0.9852_test0.9832.csv")
sub.to_csv(path, index=False)
os.makedirs(WORK_DIR, exist_ok=True); local = os.path.join(WORK_DIR, "submission.csv"); sub.to_csv(local, index=False)
print(f"saved {path}\nKaggle predictions that differ from the 0.917 soup: {(kp != k0).sum()} of {len(kp)}")
if IN_COLAB:
    from google.colab import files; files.download(local)


saved /content/drive/MyDrive/cifar10_resnet_challenge/runs/kd_sewrn_only_student_resnext29_slasd_400ep_v1/submission_soup3_shift2_flip_10x_val0.9852_test0.9832.csv
Kaggle predictions that differ from the 0.917 soup: 31 of 2000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# ---- Greedy TTA on the 0.917 SOUP: pool of views, chosen on one half of VAL, scored on the other half (cross-fitted) ----
# Paste after the standalone cell (uses soup, normalize, zoom, MEAN, STD, data). No training. Every view = itself + its flip.
import numpy as np, torch, torch.nn.functional as F
def _shift(x, dy, dx, p=4):
    if dy == 0 and dx == 0: return x
    xp = F.pad(x, (p, p, p, p), mode="reflect"); return xp[:, :, p - dy:p - dy + 32, p - dx:p - dx + 32]
def _gblur(x, s):
    r = 2; k = torch.exp(-torch.arange(-r, r + 1, device=x.device, dtype=x.dtype) ** 2 / (2 * s ** 2)); k = k / k.sum(); c = x.size(1)
    x = F.pad(x, (r, r, r, r), mode="reflect")
    return F.conv2d(F.conv2d(x, k.view(1, 1, 1, -1).repeat(c, 1, 1, 1), groups=c), k.view(1, 1, -1, 1).repeat(c, 1, 1, 1), groups=c)
def _bilat(x01, sr, ss=1.0, r=2):
    N, C, H, W = x01.shape; k = 2 * r + 1
    pt = F.unfold(F.pad(x01, (r, r, r, r), mode="reflect"), k).view(N, C, k * k, H * W); ce = x01.view(N, C, 1, H * W)
    ax = torch.arange(-r, r + 1, device=x01.device, dtype=x01.dtype)
    w = torch.exp(-(ax[:, None] ** 2 + ax[None, :] ** 2) / (2 * ss ** 2)).view(1, 1, k * k, 1) * torch.exp(-((pt - ce) ** 2).sum(1, keepdim=True) / (2 * sr ** 2))
    return ((pt * w).sum(2) / w.sum(2)).view(N, C, H, W)

VIEWS = {   # each takes uint8 images, returns normalised float images
    "orig":        lambda u: normalize(u),
    "shift +2,0":  lambda u: _shift(normalize(u), 2, 0),  "shift -2,0": lambda u: _shift(normalize(u), -2, 0),
    "shift 0,+2":  lambda u: _shift(normalize(u), 0, 2),  "shift 0,-2": lambda u: _shift(normalize(u), 0, -2),
    "blur 0.5":    lambda u: _gblur(normalize(u).float(), 0.5),
    "sharpen 0.5": lambda u: (lambda x: x + 0.5 * (x - _gblur(x, 1.0)))(normalize(u).float()),
    "bilat 0.05":  lambda u: (_bilat(u.float() / 255, 0.05) - MEAN) / STD,
    "zoom 0.875":  lambda u: zoom(normalize(u).float(), 0.875),
    "zoom 1.125":  lambda u: zoom(normalize(u).float(), 1.125),
}
@torch.no_grad()
def vprobs(m, X, name, bs=500):
    out = []
    for i in range(0, len(X), bs):
        xb = VIEWS[name](X[i:i + bs]).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            out.append((m(xb).float().softmax(1) + m(xb.flip(3)).float().softmax(1)) / 2)
    return torch.cat(out)

soup.eval(); t0 = time.time()
V = {k: {n: vprobs(soup, X, n) for n in VIEWS} for k, X in {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}.items()}
acc = lambda p, y: (p.argmax(1) == y).float().mean().item()
for n in VIEWS: print(f"  view {n:12s}: val {acc(V['val'][n], yva):.4f} | internal test {acc(V['test'][n], yte_int):.4f}")
print(f"({time.time() - t0:.0f}s)")

def greedy(ix):
    """Forward selection on VAL images ix, starting from 'orig' (= the current flip TTA); add a view only if it STRICTLY helps."""
    chosen = ["orig"]; cur = V["val"]["orig"][ix].clone(); best = acc(cur, yva[ix])
    while True:
        cands = [(acc(cur + V["val"][n][ix], yva[ix]), n) for n in VIEWS if n not in chosen]
        if not cands: break
        a, n = max(cands)
        if a <= best: break
        chosen.append(n); cur = cur + V["val"][n][ix]; best = a
    return chosen
P = lambda k, names: sum(V[k][n] for n in names)

# cross-fitted estimate: 5 random 50/50 splits of VAL, choose on one half, score on the other (both ways)
cv = []
for seed in range(5):
    g = torch.Generator().manual_seed(seed); idx = torch.randperm(len(yva), generator=g).to(device); h = len(idx) // 2
    correct = 0
    for a, b in [(idx[:h], idx[h:]), (idx[h:], idx[:h])]:
        ch = greedy(a); correct += (P("val", ch)[b].argmax(1) == yva[b]).sum().item()
    cv.append(correct / len(yva))
cv_acc = float(np.mean(cv)); flip_val = acc(V["val"]["orig"], yva); flip_test = acc(V["test"]["orig"], yte_int)
final = greedy(torch.arange(len(yva), device=device))
fin_test = acc(P("test", final), yte_int)
print(f"\nflip only (the 0.917 soup):       val {flip_val:.4f} | internal test {flip_test:.4f}")
print(f"greedy TTA, cross-fitted val:     {cv_acc:.4f}  (5 splits: {' '.join(f'{c:.4f}' for c in cv)})")
print(f"greedy TTA chosen on all of VAL:  {final} -> internal test {fin_test:.4f} ({int(round((fin_test - flip_test) * 2500)):+d} images vs flip)")
if cv_acc > flip_val + 0.002 and len(final) > 1:
    kp = P("kaggle", final).argmax(1).cpu().numpy(); k0 = V["kaggle"]["orig"].argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    path = os.path.join(RUN_DIR, f"submission_soup3_greedyTTA_{len(final)}views_cv{cv_acc:.4f}_test{fin_test:.4f}.csv"); sub.to_csv(path, index=False)
    print(f"DECISION: use it -> {path} | Kaggle predictions changed vs soup: {(kp != k0).sum()} of {len(kp)}")
else:
    print("DECISION: keep flip only (greedy TTA does not beat it on cross-fitted VAL by > 0.2 pt). No file written.")

  view orig        : val 0.9868 | internal test 0.9824
  view shift +2,0  : val 0.9860 | internal test 0.9808
  view shift -2,0  : val 0.9848 | internal test 0.9808
  view shift 0,+2  : val 0.9848 | internal test 0.9828
  view shift 0,-2  : val 0.9844 | internal test 0.9820
  view blur 0.5    : val 0.9832 | internal test 0.9792
  view sharpen 0.5 : val 0.9860 | internal test 0.9796
  view bilat 0.05  : val 0.9800 | internal test 0.9792
  view zoom 0.875  : val 0.9768 | internal test 0.9760
  view zoom 1.125  : val 0.9732 | internal test 0.9716
(9s)

flip only (the 0.917 soup):       val 0.9868 | internal test 0.9824
greedy TTA, cross-fitted val:     0.9858  (5 splits: 0.9852 0.9860 0.9860 0.9868 0.9852)
greedy TTA chosen on all of VAL:  ['orig'] -> internal test 0.9824 (+0 images vs flip)
DECISION: keep flip only (greedy TTA does not beat it on cross-fitted VAL by > 0.2 pt). No file written.


In [7]:
# ---- Shift sweep on the 0.917 SOUP: 1 / 2 / 3 px, 4 directions or 3x3 grid, all with flips (reflect padding as in training) ----
def shift(x, dy, dx, p=4):
    if dy == 0 and dx == 0: return x
    xp = F.pad(x, (p, p, p, p), mode="reflect")
    return xp[:, :, p - dy:p - dy + 32, p - dx:p - dx + 32].contiguous(memory_format=torch.channels_last)

@torch.no_grad()
def sview(m, X, dy, dx, bs=500):      # one position, averaged with its flip
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs])
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            out.append((m(shift(xb, dy, dx)).float().softmax(1) + m(shift(xb.flip(3), dy, dx)).float().softmax(1)) / 2)
    return torch.cat(out)

SW = {"flip only (current 0.917)": [(0, 0)]}
for d in (1, 2, 3):
    SW[f"{d}px, 4 directions (10x)"] = [(0, 0), (d, 0), (-d, 0), (0, d), (0, -d)]
    SW[f"{d}px, 3x3 grid (18x)"] = [(dy, dx) for dy in (-d, 0, d) for dx in (-d, 0, d)]
need = sorted({p for v in SW.values() for p in v})
soup.eval(); t0 = time.time()
S = {k: {p: sview(soup, X, *p) for p in need} for k, X in {"val": Xva, "test": Xte_int, "kaggle": Xkaggle}.items()}
acc = lambda p, y: (p.argmax(1) == y).float().mean().item()
PS = lambda k, st: sum(S[k][p] for p in SW[st])
rows = [(st, len(v) * 2, acc(PS("val", st), yva), acc(PS("test", st), yte_int)) for st, v in SW.items()]
df = pd.DataFrame(rows, columns=["setting", "views", "val", "internal_test"])
print(df.round(4).to_string(index=False)); print(f"({time.time() - t0:.0f}s)")

b0 = df.iloc[0]; c = df.iloc[1:].sort_values(["val", "views"], ascending=[False, True]).iloc[0]
print(f"\nbest on VAL: {c.setting}: val {int(round((c.val - b0.val) * 2500)):+d} images | internal {int(round((c.internal_test - b0.internal_test) * 2500)):+d} images vs flip only")
if c.val > b0.val + 0.002:
    kp = PS("kaggle", c.setting).argmax(1).cpu().numpy(); k0 = PS("kaggle", b0.setting).argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    tag = re.sub(r"[^a-z0-9]+", "_", c.setting.lower()).strip("_")
    path = os.path.join(RUN_DIR, f"submission_soup3_shiftsweep_{tag}_val{c.val:.4f}_test{c.internal_test:.4f}.csv"); sub.to_csv(path, index=False)
    print(f"DECISION: use it -> {path} | Kaggle predictions changed vs soup: {(kp != k0).sum()} of {len(kp)}")
else:
    print("DECISION: keep flip only (no shift setting beats it on VAL by > 0.2 pt). No file written.")

                  setting  views    val  internal_test
flip only (current 0.917)      2 0.9868         0.9824
  1px, 4 directions (10x)     10 0.9860         0.9840
      1px, 3x3 grid (18x)     18 0.9864         0.9832
  2px, 4 directions (10x)     10 0.9852         0.9832
      2px, 3x3 grid (18x)     18 0.9864         0.9820
  3px, 4 directions (10x)     10 0.9868         0.9844
      3px, 3x3 grid (18x)     18 0.9864         0.9836
(21s)

best on VAL: 3px, 4 directions (10x): val +0 images | internal +5 images vs flip only
DECISION: keep flip only (no shift setting beats it on VAL by > 0.2 pt). No file written.


In [8]:
st = "3px, 4 directions (10x)"
kp = PS("kaggle", st).argmax(1).cpu().numpy(); k0 = PS("kaggle", "flip only (current 0.917)").argmax(1).cpu().numpy()
sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
path = os.path.join(RUN_DIR, "submission_soup3_shift3px_4dir_flip_10x_val0.9868_test0.9844.csv"); sub.to_csv(path, index=False)
os.makedirs(WORK_DIR, exist_ok=True); local = os.path.join(WORK_DIR, "submission.csv"); sub.to_csv(local, index=False)
print(f"saved {path}\nKaggle predictions that differ from the 0.917 soup: {(kp != k0).sum()} of {len(kp)}")
if IN_COLAB:
    from google.colab import files; files.download(local)

saved /content/drive/MyDrive/cifar10_resnet_challenge/runs/kd_sewrn_only_student_resnext29_slasd_400ep_v1/submission_soup3_shift3px_4dir_flip_10x_val0.9868_test0.9844.csv
Kaggle predictions that differ from the 0.917 soup: 26 of 2000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>